In [ ]:
import json
import math
from pathlib import Path
import pandas as pd

# ==============================================================================
# 1. SETUP AND DATASET LOADING
# ==============================================================================
DATA_DIR = Path("/kaggle/input/competitions/rsna-knee-abnormality-detection")

# Load dataset
train = pd.read_csv(DATA_DIR / "train.csv")

report_col = "Report"
study_col = "StudyInstanceUID"
BATCH_SIZE = 50

# ==============================================================================
# 2. INSTRUCTIONS GENERATOR
# ==============================================================================
def get_system_instructions(batch_num, total_items):
    return f"""SYSTEM INSTRUCTIONS & DIRECTIVES:

You are an expert radiologist assistant. Your task is to analyze the provided knee radiology reports (Batch {batch_num}) and generate executable Python code that constructs a pandas DataFrame and exports it to CSV.

### DIRECTIVES:
1. Output ONLY executable Python code wrapped in a ```python ... ``` code block. No conversational text before or after.
2. The Python code MUST create a pandas DataFrame containing all extracted findings for the items in this batch.
3. First column MUST be 'SequenceID' with values from 1 to {total_items} (corresponding to the Sequence_ID in the input).
4. Second column MUST be 'StudyInstanceUID' followed by DetectedLanguage (e.g. ['Spanish' 'Dutch' 'English' 'French' 'Bulgarian' 'Turkish' 'Greek' 'Croatian'].
5. Target Columns: 'ACL', "Baker's", 'Contusion', 'Effusion', 'Fracture', 'Lateral Meniscus', 'Lateral OA', 'MCL', 'Medial Meniscus', 'Medial OA', 'PF OA', 'Synovitis'.
   - Evaluate presence/severity for all 12 targets between 0.0 (Normal/Absent) and 1.0 (Definite/Severe). Always account for negations.
6. Last column MUST be 'RelevantKeywords': Extract ONLY minimal key phrases/diagnoses for present abnormalities. Keep them extremely concise.
7. End the Python script by exporting the DataFrame: `df.to_csv('batch_{batch_num}_output.csv', index=False)`.

================================================================================
REPORTS TO PROCESS (BATCH {batch_num}):
================================================================================
"""

# ==============================================================================
# 3. BATCH GENERATION FUNCTION
# ==============================================================================
def generate_batch_prompt_files(df, batch_size=BATCH_SIZE, output_dir="prompts"):
    output_path = Path(output_dir)
    output_path.mkdir(parents=True, exist_ok=True)
    
    total_reports = len(df)
    total_batches = math.ceil(total_reports / batch_size)
    
    print(f"📊 Total reports: {total_reports} | Splitting into {total_batches} batches (Batch Size: {batch_size})...")
    
    for batch_idx in range(total_batches):
        start_idx = batch_idx * batch_size
        end_idx = start_idx + batch_size
        
        # Get batch slice
        batch_df = df.iloc[start_idx:end_idx].copy()
        
        # Format batch data with Sequence_ID starting from 1 to 50 for each batch
        reports_list = []
        for seq_id, (_, row) in enumerate(batch_df.iterrows(), start=1):
            reports_list.append({
                "Sequence_ID": seq_id,
                study_col: row[study_col],
                report_col: row[report_col]
            })
            
        # Format as JSON
        formatted_json = json.dumps(reports_list, ensure_ascii=False, indent=2)
        
        # Combine instructions and data
        batch_number = batch_idx + 1
        instructions = get_system_instructions(batch_number, len(reports_list))
        full_prompt_text = instructions + formatted_json
        
        # Save to batch file
        file_name = output_path / f"prompt_batch_{batch_number}.txt"
        with open(file_name, "w", encoding="utf-8") as f:
            f.write(full_prompt_text)
            
        print(f"  └─ Created: '{file_name}' ({len(reports_list)} reports, Sequence_ID 1 to {len(reports_list)})")
        
    print(f"\n✅ Successfully generated {total_batches} prompt file(s) in '{output_dir}/' directory.")
    print("📁 Copy-paste each file's content into ChatGPT and execute the returned Python code.")

# ==============================================================================
# 4. EXECUTION
# ==============================================================================
generate_batch_prompt_files(train, batch_size=50, output_dir="prompts")

In [ ]:
# Output as python code, batch 1

data = [
    {
        "SequenceID": 1,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10004873229099053869093324292195817260",
        "DetectedLanguage": "Spanish",
        "ACL": 0.0,
        "Baker's": 0.0,
        "Contusion": 0.0,
        "Effusion": 0.5,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 1.0,
        "Medial OA": 0.5,
        "PF OA": 0.0,
        "Synovitis": 0.0,
        "RelevantKeywords": "medial meniscus tear, subchondral avn femoral condyle, medial tibiofemoral osteoarthritis, effusion",
    },
    {
        "SequenceID": 2,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10004945927472656027199792075652399585",
        "DetectedLanguage": "Dutch",
        "ACL": 0.0,
        "Baker's": 0.0,
        "Contusion": 0.0,
        "Effusion": 0.0,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 0.0,
        "Medial OA": 0.0,
        "PF OA": 0.0,
        "Synovitis": 0.0,
        "RelevantKeywords": "suspected medial meniscal tear",
    },
    {
        "SequenceID": 3,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10009278692606631573540062909909132231",
        "DetectedLanguage": "Spanish",
        "ACL": 0.0,
        "Baker's": 0.0,
        "Contusion": 0.0,
        "Effusion": 0.0,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 0.2,
        "Medial OA": 0.5,
        "PF OA": 0.4,
        "Synovitis": 0.0,
        "RelevantKeywords": "grade 2-3 medial condyle chondropathy, grade 2 patellar chondropathy, patellar and quadricipital tendinosis",
    },
    {
        "SequenceID": 4,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10009639203170750274174707434356622764",
        "DetectedLanguage": "English",
        "ACL": 0.0,
        "Baker's": 0.0,
        "Contusion": 0.0,
        "Effusion": 0.0,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.2,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 0.0,
        "Medial OA": 0.0,
        "PF OA": 0.0,
        "Synovitis": 0.0,
        "RelevantKeywords": "suspicious incomplete discoid lateral meniscus",
    },
    {
        "SequenceID": 5,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10013663742400736029415768216599146542",
        "DetectedLanguage": "French",
        "ACL": 0.0,
        "Baker's": 0.0,
        "Contusion": 0.0,
        "Effusion": 0.0,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.2,
        "MCL": 0.2,
        "Medial Meniscus": 0.2,
        "Medial OA": 0.2,
        "PF OA": 0.7,
        "Synovitis": 0.0,
        "RelevantKeywords": "moderate to severe retropatellar chondropathy, old low grade MCL sprain, mild tibiofemoral chondropathy",
    },
    {
        "SequenceID": 6,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10018552945042470316597008531304016339",
        "DetectedLanguage": "Spanish",
        "ACL": 0.0,
        "Baker's": 0.0,
        "Contusion": 0.0,
        "Effusion": 0.0,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 0.2,
        "Medial OA": 0.0,
        "PF OA": 0.2,
        "Synovitis": 0.0,
        "RelevantKeywords": "suprapatellar fat pad edema, patellar tendinosis",
    },
    {
        "SequenceID": 7,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10018749256316287542884435387067561101",
        "DetectedLanguage": "English",
        "ACL": 0.3,
        "Baker's": 0.0,
        "Contusion": 0.0,
        "Effusion": 0.2,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 0.0,
        "Medial OA": 0.0,
        "PF OA": 0.0,
        "Synovitis": 0.0,
        "RelevantKeywords": "ACL sprain, ACL ganglion cyst, minimal joint effusion",
    },
    {
        "SequenceID": 8,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10021179701366615563817491713738306606",
        "DetectedLanguage": "Bulgarian",
        "ACL": 0.0,
        "Baker's": 1.0,
        "Contusion": 0.3,
        "Effusion": 0.3,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 0.5,
        "Medial OA": 0.5,
        "PF OA": 0.0,
        "Synovitis": 0.0,
        "RelevantKeywords": "partial tear posterior horn medial meniscus, medial osteoarthritis, Baker cyst, small effusion",
    },
    {
        "SequenceID": 9,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10021288557171866794624317186288618513",
        "DetectedLanguage": "Turkish",
        "ACL": 0.0,
        "Baker's": 0.0,
        "Contusion": 0.0,
        "Effusion": 0.2,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 0.0,
        "Medial OA": 0.0,
        "PF OA": 0.0,
        "Synovitis": 0.0,
        "RelevantKeywords": "minimal joint effusion",
    },
    {
        "SequenceID": 10,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10025765742726180988537955593315007559",
        "DetectedLanguage": "English",
        "ACL": 0.0,
        "Baker's": 0.0,
        "Contusion": 0.6,
        "Effusion": 0.7,
        "Fracture": 1.0,
        "Lateral Meniscus": 0.5,
        "Lateral OA": 0.3,
        "MCL": 0.0,
        "Medial Meniscus": 0.5,
        "Medial OA": 0.3,
        "PF OA": 0.0,
        "Synovitis": 1.0,
        "RelevantKeywords": "posterior tibial intercondylar eminence fracture, PCL avulsion and partial tear, bone contusion, osteoarthritis, meniscus degeneration/possible tear, synovitis",
    },
    {
        "SequenceID": 11,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10026410865298348818136520318965356394",
        "DetectedLanguage": "English",
        "ACL": 0.0,
        "Baker's": 0.0,
        "Contusion": 0.0,
        "Effusion": 0.3,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 1.0,
        "Medial OA": 0.7,
        "PF OA": 0.4,
        "Synovitis": 0.0,
        "RelevantKeywords": "medial compartment degeneration, degenerate tear medial meniscus, grade 2 patellar chondromalacia, mild effusion",
    },
    {
        "SequenceID": 12,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10029168398037940681915299760479434938",
        "DetectedLanguage": "Spanish",
        "ACL": 0.0,
        "Baker's": 0.0,
        "Contusion": 0.0,
        "Effusion": 0.0,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.2,
        "MCL": 0.0,
        "Medial Meniscus": 0.0,
        "Medial OA": 0.0,
        "PF OA": 0.3,
        "Synovitis": 0.0,
        "RelevantKeywords": "superficial lateral tibial chondropathy, grade 1-2 patellar chondropathy",
    },
    {
        "SequenceID": 13,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10029520033957968044048068502185417806",
        "DetectedLanguage": "English",
        "ACL": 1.0,
        "Baker's": 0.0,
        "Contusion": 0.4,
        "Effusion": 0.6,
        "Fracture": 0.0,
        "Lateral Meniscus": 1.0,
        "Lateral OA": 0.8,
        "MCL": 0.8,
        "Medial Meniscus": 1.0,
        "Medial OA": 0.5,
        "PF OA": 0.4,
        "Synovitis": 0.3,
        "RelevantKeywords": "complete tear ACL and PCL, high grade partial tear MCL, complex tears medial and lateral menisci, chondrosis tibiofemoral and PF joint, synovitis",
    },
    {
        "SequenceID": 14,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10039686090738454718354181337513293981",
        "DetectedLanguage": "Greek",
        "ACL": 0.0,
        "Baker's": 0.0,
        "Contusion": 0.0,
        "Effusion": 0.0,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 0.0,
        "Medial OA": 0.0,
        "PF OA": 0.0,
        "Synovitis": 0.0,
        "RelevantKeywords": "normal exam",
    },
    {
        "SequenceID": 15,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10048842414095994814596043262132965018",
        "DetectedLanguage": "English",
        "ACL": 0.6,
        "Baker's": 0.0,
        "Contusion": 0.7,
        "Effusion": 0.7,
        "Fracture": 1.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 0.0,
        "Medial OA": 0.0,
        "PF OA": 0.0,
        "Synovitis": 0.0,
        "RelevantKeywords": "tibial plateau fracture, fibular head fracture, ACL avulsion and grade 2 injury, traumatic lipohemarthrosis",
    },
    {
        "SequenceID": 16,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10058298717414171145622092128156266866",
        "DetectedLanguage": "Greek",
        "ACL": 0.0,
        "Baker's": 0.0,
        "Contusion": 0.0,
        "Effusion": 0.3,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 0.0,
        "Medial OA": 0.0,
        "PF OA": 0.0,
        "Synovitis": 0.0,
        "RelevantKeywords": "osteochondral loose body, patellar avulsion, small effusion, iliotibial band impingement syndrome",
    },
    {
        "SequenceID": 17,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10059200627175320481920009076697103801",
        "DetectedLanguage": "Spanish",
        "ACL": 0.0,
        "Baker's": 0.0,
        "Contusion": 0.0,
        "Effusion": 0.0,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 0.5,
        "Medial OA": 0.4,
        "PF OA": 0.4,
        "Synovitis": 1.0,
        "RelevantKeywords": "medial meniscopyth, medial femoral chondral lesion, patellar chondromalacia, synovitis, prepatellar bursitis",
    },
    {
        "SequenceID": 18,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10061120822929703004201627784850458401",
        "DetectedLanguage": "Croatian",
        "ACL": 0.3,
        "Baker's": 1.0,
        "Contusion": 0.4,
        "Effusion": 0.4,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.8,
        "Lateral OA": 0.3,
        "MCL": 0.0,
        "Medial Meniscus": 1.0,
        "Medial OA": 0.9,
        "PF OA": 0.0,
        "Synovitis": 0.0,
        "RelevantKeywords": "severe gonarthrosis, complex tear medial meniscus, suspected horizontal tear lateral meniscus, mucoid degeneration ACL, small Baker cyst",
    },
    {
        "SequenceID": 19,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10064642360075735959136394136507408269",
        "DetectedLanguage": "English",
        "ACL": 0.0,
        "Baker's": 0.0,
        "Contusion": 0.0,
        "Effusion": 0.0,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 0.0,
        "Medial OA": 0.3,
        "PF OA": 0.0,
        "Synovitis": 0.0,
        "RelevantKeywords": "PCL tear, osteochondral lesion medial femoral condyle, prepatellar subcutaneous hematoma, proximal tibial osteochondroma",
    },
    {
        "SequenceID": 20,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10066945398485042741116657288959323922",
        "DetectedLanguage": "English",
        "ACL": 0.0,
        "Baker's": 0.0,
        "Contusion": 0.0,
        "Effusion": 0.4,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 0.5,
        "Medial OA": 0.3,
        "PF OA": 0.5,
        "Synovitis": 0.0,
        "RelevantKeywords": "mild medial tibiofemoral osteoarthritis, patellofemoral chondrosis, suspicious tear medial meniscus body, knee effusion",
    },
    {
        "SequenceID": 21,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10077497528995409346950814927366582736",
        "DetectedLanguage": "English",
        "ACL": 0.3,
        "Baker's": 0.0,
        "Contusion": 0.6,
        "Effusion": 0.6,
        "Fracture": 1.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 0.2,
        "Medial OA": 0.0,
        "PF OA": 0.0,
        "Synovitis": 0.0,
        "RelevantKeywords": "PCL avulsion fracture and partial tear, grade 1 ACL injury, bone marrow edema, lipohemarthrosis, anserine bursitis",
    },
    {
        "SequenceID": 22,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10077803906105306230229342168184970097",
        "DetectedLanguage": "Greek",
        "ACL": 0.5,
        "Baker's": 0.0,
        "Contusion": 0.5,
        "Effusion": 0.6,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.2,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 0.0,
        "Medial OA": 0.0,
        "PF OA": 0.0,
        "Synovitis": 0.0,
        "RelevantKeywords": "partial ACL tear, bone bruise posterior medial tibial condyle, discoid lateral meniscus, joint effusion",
    },
    {
        "SequenceID": 23,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10081117556977868957150377234052259018",
        "DetectedLanguage": "English",
        "ACL": 1.0,
        "Baker's": 0.0,
        "Contusion": 0.0,
        "Effusion": 0.4,
        "Fracture": 0.0,
        "Lateral Meniscus": 1.0,
        "Lateral OA": 0.4,
        "MCL": 0.0,
        "Medial Meniscus": 1.0,
        "Medial OA": 0.4,
        "PF OA": 0.0,
        "Synovitis": 0.0,
        "RelevantKeywords": "chronic complete ACL tear, radial tears medial and lateral menisci, medial tibiofemoral osteoarthritis, posterolateral tibial osteochondral injury, loose body, effusion",
    },
    {
        "SequenceID": 24,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10085187975213640798228717119613397941",
        "DetectedLanguage": "English",
        "ACL": 0.0,
        "Baker's": 0.0,
        "Contusion": 0.0,
        "Effusion": 0.1,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 0.8,
        "Medial OA": 0.0,
        "PF OA": 0.0,
        "Synovitis": 0.0,
        "RelevantKeywords": "oblique tear posterior horn medial meniscus, meniscal extrusion, parameniscal cyst, quadriceps and patellar tendinosis",
    },
    {
        "SequenceID": 25,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10087350795575215080051569638065974347",
        "DetectedLanguage": "English",
        "ACL": 0.0,
        "Baker's": 0.0,
        "Contusion": 0.0,
        "Effusion": 0.0,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 0.0,
        "Medial OA": 0.0,
        "PF OA": 0.0,
        "Synovitis": 0.0,
        "RelevantKeywords": "normal exam",
    },
    {
        "SequenceID": 26,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10088120558600684332737135714636269365",
        "DetectedLanguage": "Turkish",
        "ACL": 0.0,
        "Baker's": 0.0,
        "Contusion": 0.0,
        "Effusion": 0.3,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 0.0,
        "Medial OA": 0.0,
        "PF OA": 0.4,
        "Synovitis": 0.0,
        "RelevantKeywords": "patellofemoral degeneration, mild joint fluid increase",
    },
    {
        "SequenceID": 27,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10088707526476055079083082931629995449",
        "DetectedLanguage": "Croatian",
        "ACL": 0.0,
        "Baker's": 0.0,
        "Contusion": 0.0,
        "Effusion": 0.0,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 0.0,
        "Medial OA": 0.0,
        "PF OA": 0.2,
        "Synovitis": 0.0,
        "RelevantKeywords": "grade 1b patellar chondromalacia, trochlear dysplasia",
    },
    {
        "SequenceID": 28,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10090136633727522725960302994815319097",
        "DetectedLanguage": "English",
        "ACL": 0.0,
        "Baker's": 0.0,
        "Contusion": 0.0,
        "Effusion": 0.0,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.5,
        "MCL": 0.0,
        "Medial Meniscus": 1.0,
        "Medial OA": 0.5,
        "PF OA": 0.5,
        "Synovitis": 0.0,
        "RelevantKeywords": "medial meniscus tear, tricompartmental osteoarthritis",
    },
    {
        "SequenceID": 29,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10090631811016939819371208464230332678",
        "DetectedLanguage": "English",
        "ACL": 0.0,
        "Baker's": 0.0,
        "Contusion": 0.0,
        "Effusion": 0.0,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 0.0,
        "Medial OA": 0.0,
        "PF OA": 0.0,
        "Synovitis": 0.0,
        "RelevantKeywords": "tiny chondroid lesion lateral femoral condyle, normal exam",
    },
    {
        "SequenceID": 30,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10095687747295410396510538520594649149",
        "DetectedLanguage": "Spanish",
        "ACL": 0.0,
        "Baker's": 0.0,
        "Contusion": 0.0,
        "Effusion": 0.3,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.6,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 0.0,
        "Medial OA": 0.0,
        "PF OA": 0.8,
        "Synovitis": 0.0,
        "RelevantKeywords": "marginal amputation lateral meniscus body, grade 4 focal trochlear chondropathy, mild joint effusion, quadriceps/patellar tendinosis",
    },
    {
        "SequenceID": 31,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10096063158753121251217031021181036863",
        "DetectedLanguage": "Turkish",
        "ACL": 0.0,
        "Baker's": 0.0,
        "Contusion": 0.0,
        "Effusion": 0.0,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 0.0,
        "Medial OA": 0.0,
        "PF OA": 0.0,
        "Synovitis": 0.0,
        "RelevantKeywords": "normal exam",
    },
    {
        "SequenceID": 32,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10096216489317602387766193675918767557",
        "DetectedLanguage": "Croatian",
        "ACL": 0.4,
        "Baker's": 0.0,
        "Contusion": 0.3,
        "Effusion": 0.3,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.4,
        "MCL": 0.0,
        "Medial Meniscus": 1.0,
        "Medial OA": 0.3,
        "PF OA": 0.5,
        "Synovitis": 0.0,
        "RelevantKeywords": "complex tear posterior horn medial meniscus, initial mucoid degeneration and partial ACL tear, patellofemoral and femorotibial osteoarthritis, PCL ganglion cyst",
    },
    {
        "SequenceID": 33,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10096919689272424388113819845481617364",
        "DetectedLanguage": "Turkish",
        "ACL": 0.0,
        "Baker's": 0.0,
        "Contusion": 0.0,
        "Effusion": 0.0,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 0.3,
        "Medial OA": 0.0,
        "PF OA": 0.0,
        "Synovitis": 0.0,
        "RelevantKeywords": "grade 2 medial meniscus degeneration",
    },
    {
        "SequenceID": 34,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10100718559177860518999368876392117659",
        "DetectedLanguage": "English",
        "ACL": 0.0,
        "Baker's": 0.0,
        "Contusion": 0.0,
        "Effusion": 0.0,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 0.0,
        "Medial OA": 0.0,
        "PF OA": 0.0,
        "Synovitis": 0.0,
        "RelevantKeywords": "normal exam, minimal lateral patellar tilt",
    },
    {
        "SequenceID": 35,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10112440805412493412191181061985753636",
        "DetectedLanguage": "Spanish",
        "ACL": 0.0,
        "Baker's": 0.0,
        "Contusion": 0.0,
        "Effusion": 0.0,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.6,
        "MCL": 0.0,
        "Medial Meniscus": 0.0,
        "Medial OA": 0.0,
        "PF OA": 0.4,
        "Synovitis": 0.0,
        "RelevantKeywords": "distal femoral bone infarction, grade 2-3 lateral tibial chondropathy, grade 2 patellar chondropathy, quadriceps tendinosis",
    },
    {
        "SequenceID": 36,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10118623027360228027776975311765133100",
        "DetectedLanguage": "English",
        "ACL": 0.0,
        "Baker's": 0.0,
        "Contusion": 0.0,
        "Effusion": 0.0,
        "Fracture": 0.0,
        "Lateral Meniscus": 1.0,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 0.0,
        "Medial OA": 0.0,
        "PF OA": 0.0,
        "Synovitis": 0.0,
        "RelevantKeywords": "lateral meniscus tear",
    },
    {
        "SequenceID": 37,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10120841513315425967980213160839994882",
        "DetectedLanguage": "Bulgarian",
        "ACL": 0.0,
        "Baker's": 0.0,
        "Contusion": 0.2,
        "Effusion": 0.0,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 1.0,
        "Medial OA": 0.4,
        "PF OA": 0.0,
        "Synovitis": 0.0,
        "RelevantKeywords": "radial tear posterior horn medial meniscus, initial medial gonarthrosis, intra-articular loose body, bone marrow edema",
    },
    {
        "SequenceID": 38,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10121558854084271154265851933273260339",
        "DetectedLanguage": "Turkish",
        "ACL": 0.0,
        "Baker's": 0.0,
        "Contusion": 0.0,
        "Effusion": 0.0,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.3,
        "Lateral OA": 0.0,
        "MCL": 0.3,
        "Medial Meniscus": 0.8,
        "Medial OA": 0.0,
        "PF OA": 0.0,
        "Synovitis": 0.0,
        "RelevantKeywords": "minimal horizontal tear posterior horn medial meniscus, parameniscal cyst, grade 1 MCL sprain, grade 2 lateral meniscus degeneration",
    },
    {
        "SequenceID": 39,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10123751739767586025744123799948754343",
        "DetectedLanguage": "Turkish",
        "ACL": 0.3,
        "Baker's": 0.0,
        "Contusion": 0.0,
        "Effusion": 0.6,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 1.0,
        "Medial OA": 0.0,
        "PF OA": 0.0,
        "Synovitis": 0.0,
        "RelevantKeywords": "grade 3 tear anterior horn medial meniscus, ACL degeneration, joint effusion",
    },
    {
        "SequenceID": 40,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10124063607403942330156450768015985834",
        "DetectedLanguage": "Croatian",
        "ACL": 0.0,
        "Baker's": 0.2,
        "Contusion": 0.0,
        "Effusion": 0.0,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.2,
        "MCL": 0.0,
        "Medial Meniscus": 0.3,
        "Medial OA": 0.2,
        "PF OA": 0.3,
        "Synovitis": 0.0,
        "RelevantKeywords": "mucoid degeneration posterior horn medial meniscus, grade 1 tibiofemoral chondromalacia, grade 1-2 patellar chondromalacia, minimal Baker cyst fluid",
    },
    {
        "SequenceID": 41,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10124701778993602117271554854098856833",
        "DetectedLanguage": "English",
        "ACL": 0.0,
        "Baker's": 0.3,
        "Contusion": 0.0,
        "Effusion": 0.3,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 0.0,
        "Medial OA": 0.0,
        "PF OA": 0.5,
        "Synovitis": 0.0,
        "RelevantKeywords": "retropatellar articular cartilage loss, small Baker cyst with ossific body, superolateral fat pad impingement, semimembranosus tendinosis",
    },
    {
        "SequenceID": 42,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10128954897472795975323353442305671586",
        "DetectedLanguage": "Croatian",
        "ACL": 0.0,
        "Baker's": 0.0,
        "Contusion": 0.0,
        "Effusion": 0.0,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 0.0,
        "Medial OA": 0.0,
        "PF OA": 0.0,
        "Synovitis": 0.0,
        "RelevantKeywords": "normal exam",
    },
    {
        "SequenceID": 43,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10129462959695121379466274688956171166",
        "DetectedLanguage": "Croatian",
        "ACL": 0.0,
        "Baker's": 0.5,
        "Contusion": 0.6,
        "Effusion": 0.3,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.9,
        "MCL": 0.0,
        "Medial Meniscus": 1.0,
        "Medial OA": 1.0,
        "PF OA": 1.0,
        "Synovitis": 0.0,
        "RelevantKeywords": "complex degenerative tear medial meniscus, grade 4 tricompartmental chondromalacia, patellar subluxation, popliteal cyst",
    },
    {
        "SequenceID": 44,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10132169498032330070492330913783717864",
        "DetectedLanguage": "English",
        "ACL": 0.0,
        "Baker's": 0.0,
        "Contusion": 0.0,
        "Effusion": 0.0,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 0.0,
        "Medial OA": 0.0,
        "PF OA": 0.8,
        "Synovitis": 0.0,
        "RelevantKeywords": "severe chondromalacia patella, gastrocnemius tendinosis",
    },
    {
        "SequenceID": 45,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10133843191898064967257276100779798471",
        "DetectedLanguage": "Spanish",
        "ACL": 0.0,
        "Baker's": 0.0,
        "Contusion": 0.0,
        "Effusion": 0.0,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 0.0,
        "Medial OA": 0.0,
        "PF OA": 0.5,
        "Synovitis": 0.0,
        "RelevantKeywords": "patellar osteochondral lesion, Hoffa fat pad impingement",
    },
    {
        "SequenceID": 46,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10135587809070718435691939321937314104",
        "DetectedLanguage": "Spanish",
        "ACL": 0.0,
        "Baker's": 0.0,
        "Contusion": 0.5,
        "Effusion": 0.5,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.3,
        "Lateral OA": 0.4,
        "MCL": 0.0,
        "Medial Meniscus": 1.0,
        "Medial OA": 1.0,
        "PF OA": 0.8,
        "Synovitis": 0.4,
        "RelevantKeywords": "radial and complex tear medial meniscus with extrusion, grade 4 medial tibiofemoral chondropathy, grade 3-4 patellar chondropathy, mild-moderate effusion and synovitis",
    },
    {
        "SequenceID": 47,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10136486417446492428821313562961937182",
        "DetectedLanguage": "Turkish",
        "ACL": 0.3,
        "Baker's": 0.0,
        "Contusion": 0.0,
        "Effusion": 0.0,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.2,
        "Lateral OA": 0.0,
        "MCL": 0.4,
        "Medial Meniscus": 0.3,
        "Medial OA": 0.0,
        "PF OA": 0.0,
        "Synovitis": 0.0,
        "RelevantKeywords": "grade 2 medial meniscus degeneration, grade 1 lateral meniscus degeneration, ACL myxoid degeneration, grade 2 MCL sprain",
    },
    {
        "SequenceID": 48,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10143536655047216390592811329604738314",
        "DetectedLanguage": "English",
        "ACL": 1.0,
        "Baker's": 0.0,
        "Contusion": 0.0,
        "Effusion": 0.8,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 0.8,
        "Medial OA": 0.0,
        "PF OA": 0.0,
        "Synovitis": 0.0,
        "RelevantKeywords": "complete ACL tear, horizontal tear body and posterior horn medial meniscus, large joint effusion",
    },
    {
        "SequenceID": 49,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10144735099702632881322032871591573824",
        "DetectedLanguage": "English",
        "ACL": 0.0,
        "Baker's": 0.2,
        "Contusion": 0.5,
        "Effusion": 0.0,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.7,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 0.7,
        "Medial OA": 0.0,
        "PF OA": 0.8,
        "Synovitis": 0.0,
        "RelevantKeywords": "radial tear posterior horn medial meniscus, vertical tear lateral meniscus body, severe patellar apex chondromalacia with bone edema, tiny Baker cyst",
    },
    {
        "SequenceID": 50,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10147180484860577398303524379319212586",
        "DetectedLanguage": "Turkish",
        "ACL": 0.3,
        "Baker's": 0.0,
        "Contusion": 0.5,
        "Effusion": 0.6,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.3,
        "Lateral OA": 0.5,
        "MCL": 0.3,
        "Medial Meniscus": 0.8,
        "Medial OA": 0.8,
        "PF OA": 0.5,
        "Synovitis": 0.8,
        "RelevantKeywords": "gonarthrosis, grade 3 meniscopathy and extrusion medial meniscus, ACL mucoid degeneration, grade 1 MCL sprain, suprapatellar bursitis, synovitis",
    },
]

# Create DataFrame
df = pd.DataFrame(data)

# Reorder columns explicitly as requested
target_columns = [
    "SequenceID",
    "StudyInstanceUID",
    "DetectedLanguage",
    "ACL",
    "Baker's",
    "Contusion",
    "Effusion",
    "Fracture",
    "Lateral Meniscus",
    "Lateral OA",
    "MCL",
    "Medial Meniscus",
    "Medial OA",
    "PF OA",
    "Synovitis",
    "RelevantKeywords",
]

df = df[target_columns]

# Export to CSV
df.to_csv("batch_1_output.csv", index=False)

df_cols = df.columns
df

In [ ]:
# Output as python code, batch 2

data = [
    {
        "SequenceID": 1,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10147717839726734017869499799005084725",
        "DetectedLanguage": "Turkish",
        "ACL": 0.0,
        "Baker's": 0.0,
        "Contusion": 0.0,
        "Effusion": 0.3,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.5,
        "Lateral OA": 0.0,
        "MCL": 0.2,
        "Medial Meniscus": 0.8,
        "Medial OA": 0.5,
        "PF OA": 0.9,
        "Synovitis": 0.0,
        "RelevantKeywords": "tricompartmental OA, medial meniscus tear, lateral meniscus tear, minimal effusion",
    },
    {
        "SequenceID": 2,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10148037169326559689992758180208780221",
        "DetectedLanguage": "English",
        "ACL": 0.0,
        "Baker's": 0.0,
        "Contusion": 0.0,
        "Effusion": 0.3,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 0.8,
        "Medial OA": 0.0,
        "PF OA": 0.0,
        "Synovitis": 0.0,
        "RelevantKeywords": "mild effusion, medial meniscus tear",
    },
    {
        "SequenceID": 3,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10148517526101576084961355066637531019",
        "DetectedLanguage": "Croatian",
        "ACL": 1.0,
        "Baker's": 0.0,
        "Contusion": 0.7,
        "Effusion": 0.5,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.0,
        "MCL": 0.3,
        "Medial Meniscus": 0.7,
        "Medial OA": 0.0,
        "PF OA": 0.8,
        "Synovitis": 0.0,
        "RelevantKeywords": "osteochondral lesion, chondromalacia IV, complete ACL rupture, medial meniscus tear, moderate effusion",
    },
    {
        "SequenceID": 4,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10149967031679082348901055550415519790",
        "DetectedLanguage": "Spanish",
        "ACL": 0.0,
        "Baker's": 0.0,
        "Contusion": 0.0,
        "Effusion": 0.0,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 0.0,
        "Medial OA": 0.5,
        "PF OA": 0.3,
        "Synovitis": 0.0,
        "RelevantKeywords": "chondropathy medial condyle, patellar chondropathy, patellar tendinosis",
    },
    {
        "SequenceID": 5,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10150914432481980534086434136814789205",
        "DetectedLanguage": "Bulgarian",
        "ACL": 0.0,
        "Baker's": 0.0,
        "Contusion": 0.3,
        "Effusion": 0.3,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 0.2,
        "Medial OA": 0.0,
        "PF OA": 0.6,
        "Synovitis": 0.0,
        "RelevantKeywords": "osteochondral lesion lateral tibia, patellar chondromalacia grade 3, small effusion",
    },
    {
        "SequenceID": 6,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10155611590208164695256621422536643320",
        "DetectedLanguage": "English",
        "ACL": 1.0,
        "Baker's": 0.0,
        "Contusion": 0.0,
        "Effusion": 0.0,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.8,
        "Lateral OA": 0.0,
        "MCL": 0.3,
        "Medial Meniscus": 0.9,
        "Medial OA": 0.0,
        "PF OA": 0.0,
        "Synovitis": 0.0,
        "RelevantKeywords": "bucket-handle medial meniscus tear, horizontal/radial lateral meniscus tear, meniscal cyst, complete ACL tear",
    },
    {
        "SequenceID": 7,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10157549662917062273945150928668828930",
        "DetectedLanguage": "English",
        "ACL": 0.0,
        "Baker's": 0.0,
        "Contusion": 0.0,
        "Effusion": 0.1,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.4,
        "MCL": 0.0,
        "Medial Meniscus": 0.9,
        "Medial OA": 0.4,
        "PF OA": 0.4,
        "Synovitis": 0.0,
        "RelevantKeywords": "medial meniscus complete tear, displaced meniscal fragment, tricompartmental chondrosis",
    },
    {
        "SequenceID": 8,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10158178668962379857908493434842864484",
        "DetectedLanguage": "English",
        "ACL": 0.0,
        "Baker's": 0.2,
        "Contusion": 0.0,
        "Effusion": 0.0,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.2,
        "Lateral OA": 0.3,
        "MCL": 0.0,
        "Medial Meniscus": 0.7,
        "Medial OA": 0.0,
        "PF OA": 0.3,
        "Synovitis": 0.0,
        "RelevantKeywords": "medial meniscus tear, trace Baker cyst, patellofemoral and lateral chondrosis",
    },
    {
        "SequenceID": 9,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10159551371305936155888341171022480536",
        "DetectedLanguage": "English",
        "ACL": 1.0,
        "Baker's": 0.0,
        "Contusion": 0.6,
        "Effusion": 0.6,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 0.0,
        "Medial OA": 0.0,
        "PF OA": 0.0,
        "Synovitis": 0.0,
        "RelevantKeywords": "ACL rupture, moderate effusion, bone contusions",
    },
    {
        "SequenceID": 10,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10161011217774350674534600439166319161",
        "DetectedLanguage": "English",
        "ACL": 0.0,
        "Baker's": 0.0,
        "Contusion": 0.0,
        "Effusion": 0.3,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 0.2,
        "Medial OA": 0.0,
        "PF OA": 0.0,
        "Synovitis": 0.0,
        "RelevantKeywords": "mild effusion, medial meniscus myxoid degeneration",
    },
    {
        "SequenceID": 11,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10162168547770229150261479353227398478",
        "DetectedLanguage": "Spanish",
        "ACL": 0.0,
        "Baker's": 0.0,
        "Contusion": 0.0,
        "Effusion": 0.0,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 0.9,
        "Medial OA": 0.0,
        "PF OA": 0.6,
        "Synovitis": 0.0,
        "RelevantKeywords": "complex medial meniscus tear, parameniscal cyst, patellar chondropathy grade 2-3",
    },
    {
        "SequenceID": 12,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10162720374291180196973998234794414415",
        "DetectedLanguage": "Spanish",
        "ACL": 0.0,
        "Baker's": 0.0,
        "Contusion": 0.4,
        "Effusion": 0.0,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 0.8,
        "Medial OA": 0.0,
        "PF OA": 0.0,
        "Synovitis": 0.0,
        "RelevantKeywords": "medial meniscus tear, osteochondral lesion lateral tibial plateau",
    },
    {
        "SequenceID": 13,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10165669527316708372190215034240922433",
        "DetectedLanguage": "English",
        "ACL": 0.0,
        "Baker's": 0.0,
        "Contusion": 0.0,
        "Effusion": 0.2,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 0.2,
        "Medial OA": 0.2,
        "PF OA": 0.0,
        "Synovitis": 0.0,
        "RelevantKeywords": "minor effusion, medial meniscus myxoid degeneration, mild medial cartilage thinning",
    },
    {
        "SequenceID": 14,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10170898615867673028696505248839028269",
        "DetectedLanguage": "English",
        "ACL": 0.0,
        "Baker's": 0.0,
        "Contusion": 0.5,
        "Effusion": 0.6,
        "Fracture": 0.7,
        "Lateral Meniscus": 0.8,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 0.0,
        "Medial OA": 0.0,
        "PF OA": 0.8,
        "Synovitis": 0.8,
        "RelevantKeywords": "lateral meniscus horizontal tear, osteochondral fracture patella, full thickness trochlea defect, moderate effusion, chronic reactive synovitis",
    },
    {
        "SequenceID": 15,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10181119664728335020976262955776463973",
        "DetectedLanguage": "English",
        "ACL": 0.0,
        "Baker's": 0.0,
        "Contusion": 0.5,
        "Effusion": 0.0,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 0.0,
        "Medial OA": 0.0,
        "PF OA": 0.0,
        "Synovitis": 0.0,
        "RelevantKeywords": "medial femoral condyle bone marrow edema, Hoffa fat pad edema",
    },
    {
        "SequenceID": 16,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10181291283712008544475048457589664787",
        "DetectedLanguage": "English",
        "ACL": 0.7,
        "Baker's": 0.8,
        "Contusion": 0.6,
        "Effusion": 0.5,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 0.9,
        "Medial OA": 0.3,
        "PF OA": 0.0,
        "Synovitis": 0.0,
        "RelevantKeywords": "medial meniscus complex tear, high grade partial ACL tear, Baker cyst, tibial/femoral bone bruises, effusion",
    },
    {
        "SequenceID": 17,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10182565688010898600749222617373997197",
        "DetectedLanguage": "Dutch",
        "ACL": 0.0,
        "Baker's": 0.0,
        "Contusion": 0.3,
        "Effusion": 0.1,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 0.0,
        "Medial OA": 0.3,
        "PF OA": 0.7,
        "Synovitis": 0.0,
        "RelevantKeywords": "patellofemoral cartilage disease grade 2-4, patellar tendinitis, medial femoral condyle cartilage disease grade 2",
    },
    {
        "SequenceID": 18,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10183939841890895037058579455663880292",
        "DetectedLanguage": "Turkish",
        "ACL": 0.0,
        "Baker's": 0.0,
        "Contusion": 0.0,
        "Effusion": 0.0,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.0,
        "MCL": 0.2,
        "Medial Meniscus": 0.2,
        "Medial OA": 0.0,
        "PF OA": 0.0,
        "Synovitis": 0.0,
        "RelevantKeywords": "medial meniscus degeneration, MCL edema, prepatellar edema, gastrocnemius-semimembranosus bursitis leak",
    },
    {
        "SequenceID": 19,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10189392162056736163704940747328177364",
        "DetectedLanguage": "Spanish",
        "ACL": 0.0,
        "Baker's": 0.5,
        "Contusion": 0.0,
        "Effusion": 0.3,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.0,
        "MCL": 0.6,
        "Medial Meniscus": 0.2,
        "Medial OA": 0.0,
        "PF OA": 0.0,
        "Synovitis": 0.0,
        "RelevantKeywords": "MCL grade II sprain, mild joint effusion, ruptured popliteal cyst, proximal patellar tendinosis",
    },
    {
        "SequenceID": 20,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10190506315646895064803936408630113507",
        "DetectedLanguage": "English",
        "ACL": 1.0,
        "Baker's": 0.0,
        "Contusion": 0.0,
        "Effusion": 0.1,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 0.8,
        "Medial OA": 0.0,
        "PF OA": 0.0,
        "Synovitis": 0.0,
        "RelevantKeywords": "chronic complete ACL tear, medial meniscus tear, parameniscal cyst",
    },
    {
        "SequenceID": 21,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10197515493414564227930855210542221039",
        "DetectedLanguage": "English",
        "ACL": 0.0,
        "Baker's": 0.2,
        "Contusion": 0.0,
        "Effusion": 0.1,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 0.0,
        "Medial OA": 0.0,
        "PF OA": 0.3,
        "Synovitis": 0.0,
        "RelevantKeywords": "dorsal defect of patella, chondromalacia patella, bone infarct lateral femoral condyle, trace Baker cyst",
    },
    {
        "SequenceID": 22,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10197565004992937722841731691775578390",
        "DetectedLanguage": "Spanish",
        "ACL": 0.4,
        "Baker's": 0.0,
        "Contusion": 0.0,
        "Effusion": 0.0,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 0.8,
        "Medial OA": 0.0,
        "PF OA": 0.0,
        "Synovitis": 0.0,
        "RelevantKeywords": "medial meniscus tear, ACL partial tear sequel, suprapatellar fat pad impingement",
    },
    {
        "SequenceID": 23,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10199522802568420589167284753473646773",
        "DetectedLanguage": "Spanish",
        "ACL": 0.0,
        "Baker's": 0.0,
        "Contusion": 0.0,
        "Effusion": 0.0,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 0.0,
        "Medial OA": 0.0,
        "PF OA": 0.0,
        "Synovitis": 0.0,
        "RelevantKeywords": "Hoffa fat pad impingement",
    },
    {
        "SequenceID": 24,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10201197020583611347810070328854760801",
        "DetectedLanguage": "Greek",
        "ACL": 0.7,
        "Baker's": 0.4,
        "Contusion": 0.5,
        "Effusion": 0.3,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 0.8,
        "Medial OA": 0.0,
        "PF OA": 0.0,
        "Synovitis": 0.0,
        "RelevantKeywords": "subtotal/partial ACL tear, medial meniscus tear, discoid lateral meniscus, bone bruises, Baker cyst",
    },
    {
        "SequenceID": 25,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10201660732108909756259359440298787775",
        "DetectedLanguage": "Dutch",
        "ACL": 0.0,
        "Baker's": 0.0,
        "Contusion": 0.5,
        "Effusion": 0.8,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.2,
        "Lateral OA": 0.4,
        "MCL": 0.0,
        "Medial Meniscus": 0.8,
        "Medial OA": 0.9,
        "PF OA": 0.4,
        "Synovitis": 0.8,
        "RelevantKeywords": "severe effusion, chronic synovitis, advanced medial gonarthrosis, patellofemoral osteoarthritis, medial meniscus degenerative tear",
    },
    {
        "SequenceID": 26,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10205592316907266136938526795826751183",
        "DetectedLanguage": "French",
        "ACL": 0.4,
        "Baker's": 0.0,
        "Contusion": 0.6,
        "Effusion": 0.9,
        "Fracture": 0.8,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 0.0,
        "Medial OA": 0.0,
        "PF OA": 0.4,
        "Synovitis": 0.0,
        "RelevantKeywords": "tibial spine avulsion fracture, low grade ACL tear, non-displaced fractures, LCL sprain, lipohemarthrosis",
    },
    {
        "SequenceID": 27,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10207134048598925821261298801127950516",
        "DetectedLanguage": "French",
        "ACL": 0.0,
        "Baker's": 0.0,
        "Contusion": 0.0,
        "Effusion": 0.0,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 0.0,
        "Medial OA": 0.0,
        "PF OA": 0.0,
        "Synovitis": 0.0,
        "RelevantKeywords": "normal exam",
    },
    {
        "SequenceID": 28,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10209387047693448282446870911381660585",
        "DetectedLanguage": "Greek",
        "ACL": 0.0,
        "Baker's": 0.0,
        "Contusion": 0.0,
        "Effusion": 0.0,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.2,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 0.0,
        "Medial OA": 0.0,
        "PF OA": 0.0,
        "Synovitis": 0.0,
        "RelevantKeywords": "lateral meniscus degeneration, patella maltracking, Hoffa fat pad edema",
    },
    {
        "SequenceID": 29,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10211034309703220222647821582770542129",
        "DetectedLanguage": "English",
        "ACL": 0.0,
        "Baker's": 0.2,
        "Contusion": 0.0,
        "Effusion": 0.0,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 0.0,
        "Medial OA": 0.0,
        "PF OA": 0.0,
        "Synovitis": 0.0,
        "RelevantKeywords": "trace Baker cyst, lateral patellar tilt",
    },
    {
        "SequenceID": 30,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10211815027850731240126403602239595062",
        "DetectedLanguage": "Spanish",
        "ACL": 0.0,
        "Baker's": 0.0,
        "Contusion": 0.0,
        "Effusion": 0.0,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 0.0,
        "Medial OA": 0.0,
        "PF OA": 0.0,
        "Synovitis": 0.0,
        "RelevantKeywords": "Hoffa fat pad impingement",
    },
    {
        "SequenceID": 31,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10218285921271312931891912812808052375",
        "DetectedLanguage": "Spanish",
        "ACL": 0.0,
        "Baker's": 0.0,
        "Contusion": 0.0,
        "Effusion": 0.0,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 0.2,
        "Medial OA": 0.0,
        "PF OA": 0.4,
        "Synovitis": 0.0,
        "RelevantKeywords": "medial meniscus signal alteration, patellar chondropathy, Hoffa fat pad impingement, ITB friction syndrome",
    },
    {
        "SequenceID": 32,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10218458664774126578027318307405130821",
        "DetectedLanguage": "Croatian",
        "ACL": 0.0,
        "Baker's": 0.0,
        "Contusion": 0.6,
        "Effusion": 0.5,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.2,
        "MCL": 0.6,
        "Medial Meniscus": 0.0,
        "Medial OA": 0.2,
        "PF OA": 0.2,
        "Synovitis": 0.0,
        "RelevantKeywords": "bone edema, chondromalacia grade I, joint effusion, partial MCL rupture",
    },
    {
        "SequenceID": 33,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10229223546412441924801428332647762115",
        "DetectedLanguage": "English",
        "ACL": 0.3,
        "Baker's": 0.0,
        "Contusion": 0.6,
        "Effusion": 0.5,
        "Fracture": 0.4,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.4,
        "MCL": 0.0,
        "Medial Meniscus": 0.8,
        "Medial OA": 0.4,
        "PF OA": 0.8,
        "Synovitis": 0.5,
        "RelevantKeywords": "medial meniscus horizontal tear, tricompartmental OA, bone marrow edema, suspicious patellar fracture, low grade ACL/PCL partial tears, effusion",
    },
    {
        "SequenceID": 34,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10239418182819270860819533058081041576",
        "DetectedLanguage": "Spanish",
        "ACL": 0.0,
        "Baker's": 0.0,
        "Contusion": 0.5,
        "Effusion": 0.4,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 0.0,
        "Medial OA": 0.0,
        "PF OA": 0.0,
        "Synovitis": 0.0,
        "RelevantKeywords": "tibial bone contusion, effusion",
    },
    {
        "SequenceID": 35,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10240166440044087554182120832488098189",
        "DetectedLanguage": "Greek",
        "ACL": 0.0,
        "Baker's": 0.0,
        "Contusion": 0.6,
        "Effusion": 0.5,
        "Fracture": 0.6,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 0.8,
        "Medial OA": 0.9,
        "PF OA": 0.0,
        "Synovitis": 0.0,
        "RelevantKeywords": "degenerative medial meniscus tear, severe medial osteoarthritis, insufficiency microfracture, ACL myxoid degeneration, moderate effusion",
    },
    {
        "SequenceID": 36,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10240237086492917349933491458808329326",
        "DetectedLanguage": "Bulgarian",
        "ACL": 0.0,
        "Baker's": 0.0,
        "Contusion": 0.0,
        "Effusion": 0.0,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 0.2,
        "Medial OA": 0.2,
        "PF OA": 0.0,
        "Synovitis": 0.0,
        "RelevantKeywords": "medial meniscus degenerative changes, subchondral cysts tibia, Hoffa fat pad edema",
    },
    {
        "SequenceID": 37,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10241639721703882165429374175491428414",
        "DetectedLanguage": "Turkish",
        "ACL": 0.0,
        "Baker's": 0.0,
        "Contusion": 0.0,
        "Effusion": 0.0,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 0.0,
        "Medial OA": 0.0,
        "PF OA": 0.0,
        "Synovitis": 0.0,
        "RelevantKeywords": "normal exam",
    },
    {
        "SequenceID": 38,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10243300429607935614655219482457289307",
        "DetectedLanguage": "German",
        "ACL": 0.0,
        "Baker's": 0.0,
        "Contusion": 0.0,
        "Effusion": 0.0,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 0.2,
        "Medial OA": 0.3,
        "PF OA": 0.0,
        "Synovitis": 0.0,
        "RelevantKeywords": "superficial chondropathy medial femoral condyle, medial meniscus mucoid signal alteration",
    },
    {
        "SequenceID": 39,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10247930792182001003497189021027466833",
        "DetectedLanguage": "English",
        "ACL": 0.0,
        "Baker's": 0.2,
        "Contusion": 0.0,
        "Effusion": 0.2,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.6,
        "MCL": 0.0,
        "Medial Meniscus": 0.2,
        "Medial OA": 0.0,
        "PF OA": 0.8,
        "Synovitis": 0.0,
        "RelevantKeywords": "patellofemoral maltracking, high-grade patellofemoral chondrosis, moderate lateral compartment chondrosis, low-grade chondroid lesion, trace Baker cyst",
    },
    {
        "SequenceID": 40,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10252121617380434672930525780360219531",
        "DetectedLanguage": "English",
        "ACL": 0.0,
        "Baker's": 0.0,
        "Contusion": 0.0,
        "Effusion": 0.6,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 0.0,
        "Medial OA": 0.0,
        "PF OA": 0.0,
        "Synovitis": 0.8,
        "RelevantKeywords": "bilateral joint effusion, enhancing synovitis, inflammatory arthritis",
    },
    {
        "SequenceID": 41,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10252163031705285141538200795896549203",
        "DetectedLanguage": "Spanish",
        "ACL": 0.0,
        "Baker's": 0.0,
        "Contusion": 0.0,
        "Effusion": 0.0,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.7,
        "MCL": 0.0,
        "Medial Meniscus": 0.2,
        "Medial OA": 0.4,
        "PF OA": 0.9,
        "Synovitis": 0.0,
        "RelevantKeywords": "chondropathy grade 2-3 lateral femorotibial, grade 2 medial femoral condyle, grade 4 patella, patellar tendinosis",
    },
    {
        "SequenceID": 42,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10254828365033984708960944586564394286",
        "DetectedLanguage": "English",
        "ACL": 0.0,
        "Baker's": 0.0,
        "Contusion": 0.0,
        "Effusion": 0.0,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 0.0,
        "Medial OA": 0.0,
        "PF OA": 0.0,
        "Synovitis": 0.0,
        "RelevantKeywords": "normal exam",
    },
    {
        "SequenceID": 43,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10255067212695306715722782826234429766",
        "DetectedLanguage": "English",
        "ACL": 0.0,
        "Baker's": 0.0,
        "Contusion": 0.0,
        "Effusion": 0.2,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 0.8,
        "Medial OA": 0.0,
        "PF OA": 0.0,
        "Synovitis": 0.0,
        "RelevantKeywords": "medial meniscus tear, ganglion cyst posterior to PCL, minimal effusion",
    },
    {
        "SequenceID": 44,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10256441455107860474283716992657597873",
        "DetectedLanguage": "Turkish",
        "ACL": 0.2,
        "Baker's": 0.0,
        "Contusion": 0.5,
        "Effusion": 0.3,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.2,
        "Lateral OA": 0.0,
        "MCL": 0.3,
        "Medial Meniscus": 0.8,
        "Medial OA": 0.5,
        "PF OA": 0.4,
        "Synovitis": 0.0,
        "RelevantKeywords": "medial meniscus horizontal tear and extrusion, grade II patellar chondromalacia, tibial bone marrow edema, joint space narrowing",
    },
    {
        "SequenceID": 45,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10261794102024899290234062909324134903",
        "DetectedLanguage": "German",
        "ACL": 0.0,
        "Baker's": 0.0,
        "Contusion": 0.0,
        "Effusion": 0.0,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 0.0,
        "Medial OA": 0.0,
        "PF OA": 0.0,
        "Synovitis": 0.0,
        "RelevantKeywords": "normal exam",
    },
    {
        "SequenceID": 46,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10262024906324456863812446480254600405",
        "DetectedLanguage": "Turkish",
        "ACL": 0.0,
        "Baker's": 0.7,
        "Contusion": 0.0,
        "Effusion": 0.3,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.2,
        "Lateral OA": 0.0,
        "MCL": 0.2,
        "Medial Meniscus": 0.8,
        "Medial OA": 0.5,
        "PF OA": 0.0,
        "Synovitis": 0.0,
        "RelevantKeywords": "medial meniscus radial tear and extrusion, Baker cyst, medial compartment joint space narrowing, prepatellar bursitis",
    },
    {
        "SequenceID": 47,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10262784969531507133353458615303007839",
        "DetectedLanguage": "English",
        "ACL": 0.0,
        "Baker's": 0.0,
        "Contusion": 0.0,
        "Effusion": 0.0,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.6,
        "MCL": 0.0,
        "Medial Meniscus": 0.9,
        "Medial OA": 0.0,
        "PF OA": 0.6,
        "Synovitis": 0.0,
        "RelevantKeywords": "medial meniscus bucket handle tear, OA lateral and patellofemoral compartments",
    },
    {
        "SequenceID": 48,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10271567891603078238024189515805908561",
        "DetectedLanguage": "English",
        "ACL": 0.3,
        "Baker's": 0.0,
        "Contusion": 0.5,
        "Effusion": 0.5,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.8,
        "Lateral OA": 0.6,
        "MCL": 0.0,
        "Medial Meniscus": 0.8,
        "Medial OA": 0.6,
        "PF OA": 0.6,
        "Synovitis": 0.6,
        "RelevantKeywords": "osteoarthritis, medial and lateral meniscal tears, osteonecrosis tibia, synovitis, grade 1 ACL injury",
    },
    {
        "SequenceID": 49,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10273861326753707436672556008991334542",
        "DetectedLanguage": "Spanish",
        "ACL": 0.0,
        "Baker's": 0.0,
        "Contusion": 0.0,
        "Effusion": 0.0,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 0.2,
        "Medial OA": 0.0,
        "PF OA": 0.0,
        "Synovitis": 0.0,
        "RelevantKeywords": "patellar and quadriceps tendinosis, prepatellar aponeurosis edema",
    },
    {
        "SequenceID": 50,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10274314637471308855247991547659839029",
        "DetectedLanguage": "Spanish",
        "ACL": 0.0,
        "Baker's": 0.0,
        "Contusion": 0.0,
        "Effusion": 0.6,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 0.0,
        "Medial OA": 0.5,
        "PF OA": 0.6,
        "Synovitis": 0.6,
        "RelevantKeywords": "effusion with synovitis, medial chondropathy, patellofemoral OA, posterior synovial cyst",
    },
]

df = pd.DataFrame(data)
df = df[df_cols]
df.to_csv("batch_2_output.csv", index=False)
df

In [ ]:
# Output as code, batch 3

data = [
    {
        "SequenceID": 1,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10274956886032850418742861294277331382",
        "DetectedLanguage": "English",
        "ACL": 0.0,
        "Baker's": 0.5,
        "Contusion": 0.5,
        "Effusion": 0.0,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 0.0,
        "Medial OA": 0.0,
        "PF OA": 0.8,
        "Synovitis": 0.0,
        "RelevantKeywords": "partially ruptured Baker's cyst, subchondral bone marrow edema, patellofemoral chondrosis",
    },
    {
        "SequenceID": 2,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10275265831651276229168539541982478639",
        "DetectedLanguage": "Spanish",
        "ACL": 0.0,
        "Baker's": 0.0,
        "Contusion": 0.2,
        "Effusion": 0.5,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.8,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 0.6,
        "Medial OA": 0.4,
        "PF OA": 0.6,
        "Synovitis": 0.3,
        "RelevantKeywords": "rotura menisco lateral, probable rotura menisco medial, condropatia femoral medial, condropatia troclea femoral, derrame articular, sinovitis",
    },
    {
        "SequenceID": 3,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10284263494252700630857173843779724077",
        "DetectedLanguage": "English",
        "ACL": 0.8,
        "Baker's": 0.0,
        "Contusion": 0.5,
        "Effusion": 0.7,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.8,
        "Lateral OA": 0.8,
        "MCL": 0.0,
        "Medial Meniscus": 0.3,
        "Medial OA": 0.0,
        "PF OA": 0.0,
        "Synovitis": 0.8,
        "RelevantKeywords": "ACL tear, lateral meniscus tear, medial meniscus degeneration, osteoarthritis lateral compartment, joint effusion, synovitis",
    },
    {
        "SequenceID": 4,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10284446538566255572004044331831825072",
        "DetectedLanguage": "English",
        "ACL": 0.0,
        "Baker's": 0.0,
        "Contusion": 0.0,
        "Effusion": 0.5,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 0.8,
        "Medial OA": 0.2,
        "PF OA": 0.0,
        "Synovitis": 0.0,
        "RelevantKeywords": "moderate effusion, horizontal tear medial meniscus, mild medial cartilage thinning",
    },
    {
        "SequenceID": 5,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10287488031206457680758379452936914669",
        "DetectedLanguage": "Dutch",
        "ACL": 0.0,
        "Baker's": 0.0,
        "Contusion": 0.0,
        "Effusion": 0.1,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 0.0,
        "Medial OA": 0.0,
        "PF OA": 0.3,
        "Synovitis": 0.0,
        "RelevantKeywords": "patellair kraakbeenlijden graad 2, minimaal gewrichtsvocht",
    },
    {
        "SequenceID": 6,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10299396087685174806129850512856451005",
        "DetectedLanguage": "Greek",
        "ACL": 1.0,
        "Baker's": 0.0,
        "Contusion": 0.0,
        "Effusion": 0.2,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.0,
        "MCL": 1.0,
        "Medial Meniscus": 0.0,
        "Medial OA": 0.0,
        "PF OA": 0.0,
        "Synovitis": 0.0,
        "RelevantKeywords": "πλήρης ρήξη πρόσθιου χιαστού, ολική ρήξη έσω πλαγίου συνδέσμου, υγρό ενδοαρθρικά",
    },
    {
        "SequenceID": 7,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10299618611626190927173051117228647808",
        "DetectedLanguage": "Spanish",
        "ACL": 0.0,
        "Baker's": 0.0,
        "Contusion": 0.0,
        "Effusion": 0.0,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 0.0,
        "Medial OA": 0.0,
        "PF OA": 0.0,
        "Synovitis": 0.0,
        "RelevantKeywords": "sin anomalias",
    },
    {
        "SequenceID": 8,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10301089872627496769249366231947887097",
        "DetectedLanguage": "Bulgarian",
        "ACL": 0.0,
        "Baker's": 0.0,
        "Contusion": 0.0,
        "Effusion": 0.5,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 0.3,
        "Medial OA": 0.0,
        "PF OA": 0.0,
        "Synovitis": 0.0,
        "RelevantKeywords": "ставен излив, частична увреда заден рог медиален менискус",
    },
    {
        "SequenceID": 9,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10304242666970626179776896768140580106",
        "DetectedLanguage": "English",
        "ACL": 0.0,
        "Baker's": 0.0,
        "Contusion": 0.0,
        "Effusion": 0.0,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 0.0,
        "Medial OA": 0.0,
        "PF OA": 0.0,
        "Synovitis": 0.0,
        "RelevantKeywords": "lateral patellofemoral maltracking, superolateral Hoffa's fat pad edema",
    },
    {
        "SequenceID": 10,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10306159113324811538703788080836752052",
        "DetectedLanguage": "English",
        "ACL": 0.0,
        "Baker's": 0.0,
        "Contusion": 0.5,
        "Effusion": 0.3,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.2,
        "Lateral OA": 0.2,
        "MCL": 0.0,
        "Medial Meniscus": 1.0,
        "Medial OA": 0.9,
        "PF OA": 0.8,
        "Synovitis": 0.0,
        "RelevantKeywords": "complete tearing medial meniscus, tricompartmental osteoarthritis, joint effusion, bone marrow edema",
    },
    {
        "SequenceID": 11,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10309477218703786877612946261946268132",
        "DetectedLanguage": "Turkish",
        "ACL": 0.2,
        "Baker's": 0.0,
        "Contusion": 0.5,
        "Effusion": 0.0,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 0.0,
        "Medial OA": 0.0,
        "PF OA": 0.4,
        "Synovitis": 0.0,
        "RelevantKeywords": "ön çapraz bağda sinyal artışı, grade 2 kondromalazi patella, subkondral kemik iliği ödemi",
    },
    {
        "SequenceID": 12,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10310831394568337664174691281092566272",
        "DetectedLanguage": "Bulgarian",
        "ACL": 0.0,
        "Baker's": 0.0,
        "Contusion": 0.0,
        "Effusion": 0.0,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 0.3,
        "Medial OA": 0.0,
        "PF OA": 0.0,
        "Synovitis": 0.0,
        "RelevantKeywords": "лезия заден рог медиален менискус grade 2a",
    },
    {
        "SequenceID": 13,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10312350714164739081126996907250465738",
        "DetectedLanguage": "Turkish",
        "ACL": 0.0,
        "Baker's": 0.0,
        "Contusion": 0.0,
        "Effusion": 0.0,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 0.8,
        "Medial OA": 0.0,
        "PF OA": 0.0,
        "Synovitis": 0.0,
        "RelevantKeywords": "medial menisküste horizontal yırtık, parameniskal kist",
    },
    {
        "SequenceID": 14,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10314240078685804110739316196999571466",
        "DetectedLanguage": "English",
        "ACL": 0.0,
        "Baker's": 0.7,
        "Contusion": 0.5,
        "Effusion": 0.7,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.9,
        "Lateral OA": 0.9,
        "MCL": 0.0,
        "Medial Meniscus": 0.0,
        "Medial OA": 0.4,
        "PF OA": 0.5,
        "Synovitis": 0.0,
        "RelevantKeywords": "multidirection tear lateral meniscus, tricompartmental chondropathy, popliteal cyst, joint effusion, intra-articular body",
    },
    {
        "SequenceID": 15,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10318366890716843554246682186865048846",
        "DetectedLanguage": "German",
        "ACL": 0.0,
        "Baker's": 0.0,
        "Contusion": 0.6,
        "Effusion": 0.7,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.2,
        "Lateral OA": 0.2,
        "MCL": 0.0,
        "Medial Meniscus": 0.8,
        "Medial OA": 0.8,
        "PF OA": 0.5,
        "Synovitis": 0.0,
        "RelevantKeywords": "komplexe Rissbildung Innenmeniskushinterhorn, Osteoarthrose medial, subchondrales Ödem, Gelenkerguss, retropatellare Chondropathie",
    },
    {
        "SequenceID": 16,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10321461243762453573005633063753741725",
        "DetectedLanguage": "Turkish",
        "ACL": 0.4,
        "Baker's": 0.0,
        "Contusion": 0.0,
        "Effusion": 0.2,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 0.2,
        "Medial OA": 0.0,
        "PF OA": 0.0,
        "Synovitis": 0.0,
        "RelevantKeywords": "ACL grade II sprain, medial menisküs grade I dejenerasyon, minimal efüzyon",
    },
    {
        "SequenceID": 17,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10321556388278642027434955207617064548",
        "DetectedLanguage": "Croatian",
        "ACL": 0.0,
        "Baker's": 0.0,
        "Contusion": 0.0,
        "Effusion": 0.0,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.0,
        "MCL": 0.5,
        "Medial Meniscus": 0.0,
        "Medial OA": 0.0,
        "PF OA": 0.0,
        "Synovitis": 0.0,
        "RelevantKeywords": "parcijalna ruptura MCL",
    },
    {
        "SequenceID": 18,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10321702681363532511953946967334431952",
        "DetectedLanguage": "Bulgarian",
        "ACL": 0.0,
        "Baker's": 0.0,
        "Contusion": 0.0,
        "Effusion": 0.0,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 0.4,
        "Medial OA": 0.0,
        "PF OA": 0.0,
        "Synovitis": 0.0,
        "RelevantKeywords": "лезия заден рог медиален менискус 2b стадий",
    },
    {
        "SequenceID": 19,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10322724865012212544754724404226583315",
        "DetectedLanguage": "Croatian",
        "ACL": 0.3,
        "Baker's": 0.5,
        "Contusion": 0.6,
        "Effusion": 0.5,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.2,
        "Lateral OA": 0.8,
        "MCL": 0.2,
        "Medial Meniscus": 0.8,
        "Medial OA": 0.8,
        "PF OA": 0.8,
        "Synovitis": 0.0,
        "RelevantKeywords": "osteoartritis, subhondralni edem, horizontalna ruptura medijalnog meniska, Baker cista, zglobni izljev",
    },
    {
        "SequenceID": 20,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10322853084122920408377349136830993432",
        "DetectedLanguage": "English",
        "ACL": 0.0,
        "Baker's": 0.0,
        "Contusion": 0.0,
        "Effusion": 0.2,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 0.8,
        "Medial OA": 0.3,
        "PF OA": 0.0,
        "Synovitis": 0.0,
        "RelevantKeywords": "tearing posterior root medial meniscus, mild knee effusion, mild medial compartment cartilage thinning",
    },
    {
        "SequenceID": 21,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10326352669696598934883805147731461341",
        "DetectedLanguage": "Bulgarian",
        "ACL": 0.3,
        "Baker's": 0.0,
        "Contusion": 0.0,
        "Effusion": 0.0,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 0.3,
        "Medial OA": 0.0,
        "PF OA": 0.0,
        "Synovitis": 0.0,
        "RelevantKeywords": "дегенеративна промяна медиален менискус, менискална киста, мукоидна дегенерация ACL",
    },
    {
        "SequenceID": 22,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10327608571317884761792196888985963332",
        "DetectedLanguage": "English",
        "ACL": 0.0,
        "Baker's": 0.0,
        "Contusion": 0.0,
        "Effusion": 0.0,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 0.0,
        "Medial OA": 0.0,
        "PF OA": 0.0,
        "Synovitis": 0.0,
        "RelevantKeywords": "normal exam",
    },
    {
        "SequenceID": 23,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10330805760778577501878048995484987687",
        "DetectedLanguage": "Bulgarian",
        "ACL": 0.0,
        "Baker's": 0.0,
        "Contusion": 0.0,
        "Effusion": 0.5,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 0.3,
        "Medial OA": 0.0,
        "PF OA": 0.0,
        "Synovitis": 0.0,
        "RelevantKeywords": "лезия медиален менискус 2а, ставен излив",
    },
    {
        "SequenceID": 24,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10336221220793312503236159680373829726",
        "DetectedLanguage": "English",
        "ACL": 0.0,
        "Baker's": 0.0,
        "Contusion": 0.0,
        "Effusion": 0.5,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 0.0,
        "Medial OA": 0.0,
        "PF OA": 0.0,
        "Synovitis": 0.0,
        "RelevantKeywords": "joint effusion, suprapatellar bursa fluid",
    },
    {
        "SequenceID": 25,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10336300544286006569730381727202519821",
        "DetectedLanguage": "English",
        "ACL": 0.0,
        "Baker's": 0.3,
        "Contusion": 0.8,
        "Effusion": 0.5,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.0,
        "MCL": 0.4,
        "Medial Meniscus": 0.8,
        "Medial OA": 0.8,
        "PF OA": 0.8,
        "Synovitis": 0.5,
        "RelevantKeywords": "unstable horizontal tear medial meniscus, medial tibial contusion, MCL sprain, joint effusion, patellofemoral chondrosis, medial condyle chondrosis",
    },
    {
        "SequenceID": 26,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10339654093023509742663983374586998753",
        "DetectedLanguage": "Greek",
        "ACL": 0.0,
        "Baker's": 0.0,
        "Contusion": 0.0,
        "Effusion": 0.6,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 0.3,
        "Medial OA": 0.0,
        "PF OA": 0.4,
        "Synovitis": 0.0,
        "RelevantKeywords": "εκφύλιση έσω μηνίσκου, αρθρικός χόνδρος επιγονατίδας ICRS grade II, συλλογή υγρού",
    },
    {
        "SequenceID": 27,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10348611327224519953673858098392135936",
        "DetectedLanguage": "English",
        "ACL": 0.0,
        "Baker's": 0.0,
        "Contusion": 0.0,
        "Effusion": 0.2,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 0.2,
        "Medial OA": 0.0,
        "PF OA": 0.0,
        "Synovitis": 0.0,
        "RelevantKeywords": "mild knee effusion, medial meniscus myxoid degeneration",
    },
    {
        "SequenceID": 28,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10349368332722015923160611226877741306",
        "DetectedLanguage": "Spanish",
        "ACL": 0.0,
        "Baker's": 0.0,
        "Contusion": 0.0,
        "Effusion": 0.0,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 0.0,
        "Medial OA": 0.0,
        "PF OA": 0.0,
        "Synovitis": 0.0,
        "RelevantKeywords": "sindrome de friccion de Hoffa",
    },
    {
        "SequenceID": 29,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10351168527688244230248760520733020605",
        "DetectedLanguage": "English",
        "ACL": 0.2,
        "Baker's": 0.3,
        "Contusion": 0.0,
        "Effusion": 0.5,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.3,
        "MCL": 0.0,
        "Medial Meniscus": 0.3,
        "Medial OA": 0.3,
        "PF OA": 0.8,
        "Synovitis": 0.5,
        "RelevantKeywords": "osteoarthritis, patellofemoral chondromalacia grade 3-4, joint effusion, synovitis, Baker's cyst, ACL degeneration",
    },
    {
        "SequenceID": 30,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10351460911387106879135924409514835385",
        "DetectedLanguage": "Spanish",
        "ACL": 0.0,
        "Baker's": 0.0,
        "Contusion": 0.0,
        "Effusion": 0.0,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 0.0,
        "Medial OA": 0.0,
        "PF OA": 0.0,
        "Synovitis": 0.0,
        "RelevantKeywords": "sin anomalias",
    },
    {
        "SequenceID": 31,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10352011938574988727803498915608467179",
        "DetectedLanguage": "English",
        "ACL": 0.0,
        "Baker's": 0.0,
        "Contusion": 0.0,
        "Effusion": 0.1,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 0.0,
        "Medial OA": 0.0,
        "PF OA": 0.0,
        "Synovitis": 0.0,
        "RelevantKeywords": "minimal PF joint effusion",
    },
    {
        "SequenceID": 32,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10352044313309892076985642485759368197",
        "DetectedLanguage": "Croatian",
        "ACL": 0.3,
        "Baker's": 0.0,
        "Contusion": 0.7,
        "Effusion": 0.5,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.8,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 0.0,
        "Medial OA": 0.0,
        "PF OA": 0.0,
        "Synovitis": 0.0,
        "RelevantKeywords": "ruptura prednjeg roga lateralnog meniska, kostani edem tibijalnog platoa, distenzija ACL, izljev u koljenu",
    },
    {
        "SequenceID": 33,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10354576637002647863048798607011636355",
        "DetectedLanguage": "English",
        "ACL": 0.0,
        "Baker's": 0.0,
        "Contusion": 0.0,
        "Effusion": 0.0,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 0.0,
        "Medial OA": 0.0,
        "PF OA": 0.0,
        "Synovitis": 0.0,
        "RelevantKeywords": "healed nonossifying fibroma",
    },
    {
        "SequenceID": 34,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10361030765327974510262894062047216433",
        "DetectedLanguage": "Spanish",
        "ACL": 0.0,
        "Baker's": 0.0,
        "Contusion": 0.0,
        "Effusion": 0.0,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 0.5,
        "Medial OA": 0.0,
        "PF OA": 0.8,
        "Synovitis": 0.0,
        "RelevantKeywords": "lesion meniscocapsular menisco medial, condropatia grado 4 rotula, tendinosis",
    },
    {
        "SequenceID": 35,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10370612191212456931506340727148605247",
        "DetectedLanguage": "English",
        "ACL": 0.0,
        "Baker's": 0.8,
        "Contusion": 0.0,
        "Effusion": 0.6,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.8,
        "Lateral OA": 0.4,
        "MCL": 0.0,
        "Medial Meniscus": 0.8,
        "Medial OA": 0.4,
        "PF OA": 0.0,
        "Synovitis": 0.6,
        "RelevantKeywords": "Baker's cyst, medial meniscus tear, lateral meniscus tear, joint effusion, synovitis, osteoarthritis",
    },
    {
        "SequenceID": 36,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10371162371589200772388569623043753617",
        "DetectedLanguage": "German",
        "ACL": 0.3,
        "Baker's": 0.0,
        "Contusion": 0.6,
        "Effusion": 0.2,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.6,
        "MCL": 0.3,
        "Medial Meniscus": 0.8,
        "Medial OA": 0.8,
        "PF OA": 0.4,
        "Synovitis": 0.0,
        "RelevantKeywords": "Chondropathie femorotibial medial und lateral, Innenmeniskus Rissbildung, Gelenkerguss, subchondrales Ödem, mukoide Degeneration ACL",
    },
    {
        "SequenceID": 37,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10374791734880310643587680407133671218",
        "DetectedLanguage": "Dutch",
        "ACL": 0.0,
        "Baker's": 0.7,
        "Contusion": 0.0,
        "Effusion": 0.4,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.8,
        "Lateral OA": 0.5,
        "MCL": 0.0,
        "Medial Meniscus": 0.8,
        "Medial OA": 0.4,
        "PF OA": 0.9,
        "Synovitis": 0.0,
        "RelevantKeywords": "scheur mediale meniscus, scheur laterale meniscus, femoropatellair kraakbeenverlies, geruptureerde Baker cyste",
    },
    {
        "SequenceID": 38,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10378535286770237477615181844634903700",
        "DetectedLanguage": "Bulgarian",
        "ACL": 0.0,
        "Baker's": 0.7,
        "Contusion": 0.3,
        "Effusion": 0.2,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.8,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 0.3,
        "Medial OA": 0.0,
        "PF OA": 0.6,
        "Synovitis": 0.0,
        "RelevantKeywords": "хоризонтална руптура латерален менискус, дегенерация медиален менискус, увреда LCL, хондромалация патела grade 3, Бекерова киста, минимален излив",
    },
    {
        "SequenceID": 39,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10382074558506016193900889176239337801",
        "DetectedLanguage": "English",
        "ACL": 0.0,
        "Baker's": 0.0,
        "Contusion": 0.5,
        "Effusion": 0.0,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 0.0,
        "Medial OA": 0.0,
        "PF OA": 0.6,
        "Synovitis": 0.0,
        "RelevantKeywords": "patellofemoral osteoarthritis, patellar cartilage loss, subchondral marrow edema",
    },
    {
        "SequenceID": 40,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10386894500329189457070808193345170500",
        "DetectedLanguage": "English",
        "ACL": 0.0,
        "Baker's": 0.2,
        "Contusion": 0.0,
        "Effusion": 0.1,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 0.0,
        "Medial OA": 0.0,
        "PF OA": 0.0,
        "Synovitis": 0.0,
        "RelevantKeywords": "small popliteal cyst, trace effusion, Hoffa's fat pad impingement",
    },
    {
        "SequenceID": 41,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10387227472110643414333498953513810066",
        "DetectedLanguage": "Croatian",
        "ACL": 0.5,
        "Baker's": 0.0,
        "Contusion": 0.8,
        "Effusion": 0.5,
        "Fracture": 0.3,
        "Lateral Meniscus": 0.4,
        "Lateral OA": 0.2,
        "MCL": 0.3,
        "Medial Meniscus": 0.2,
        "Medial OA": 0.2,
        "PF OA": 0.2,
        "Synovitis": 0.0,
        "RelevantKeywords": "kostani edem, osteohondralna lezija, parcijalna ruptura ACL, intrasupstancijska ruptura lateralnog meniska, zglobni izljev",
    },
    {
        "SequenceID": 42,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10387383212921524066516723428665512341",
        "DetectedLanguage": "Croatian",
        "ACL": 0.3,
        "Baker's": 0.0,
        "Contusion": 0.3,
        "Effusion": 0.4,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 0.3,
        "Medial OA": 0.7,
        "PF OA": 0.4,
        "Synovitis": 0.3,
        "RelevantKeywords": "osteoartritis medijalnog kompartmenta, subhondralni edem, mukoidna degeneracija ACL i PCL, mukoidna degeneracija medijalnog meniska, izljev, sinovitis",
    },
    {
        "SequenceID": 43,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10387876708310019345913209598612198432",
        "DetectedLanguage": "Spanish",
        "ACL": 0.5,
        "Baker's": 0.4,
        "Contusion": 0.3,
        "Effusion": 0.2,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 0.8,
        "Medial OA": 0.0,
        "PF OA": 0.2,
        "Synovitis": 0.0,
        "RelevantKeywords": "rotura parcial LCA, rotura compleja menisco medial, edema oseo, condropatia rotuliana, ganglion quistico, leve derrame",
    },
    {
        "SequenceID": 44,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10391178814963564201551003358372148419",
        "DetectedLanguage": "Turkish",
        "ACL": 0.0,
        "Baker's": 0.3,
        "Contusion": 0.0,
        "Effusion": 0.5,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 0.8,
        "Medial OA": 0.8,
        "PF OA": 0.0,
        "Synovitis": 0.0,
        "RelevantKeywords": "medyal femorotibyal osteoartrit, medyal menisküs kompleks dejeneratif yırtık, eklem içi sıvı artışı, gastrokinemyus-semimembranozus bursiti",
    },
    {
        "SequenceID": 45,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10395211177230760565674811878437948933",
        "DetectedLanguage": "Turkish",
        "ACL": 0.0,
        "Baker's": 0.5,
        "Contusion": 0.0,
        "Effusion": 0.2,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.0,
        "MCL": 0.3,
        "Medial Meniscus": 0.6,
        "Medial OA": 0.8,
        "PF OA": 0.6,
        "Synovitis": 0.5,
        "RelevantKeywords": "gonartroz, medial menisküs ekstrüzyon ve dejenerasyon, MCL sprain, Baker kisti, sinovit, osteokondral defekt",
    },
    {
        "SequenceID": 46,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10401813253995823397883215400000450724",
        "DetectedLanguage": "Croatian",
        "ACL": 0.2,
        "Baker's": 0.0,
        "Contusion": 0.0,
        "Effusion": 0.0,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.2,
        "MCL": 0.0,
        "Medial Meniscus": 0.0,
        "Medial OA": 0.2,
        "PF OA": 0.0,
        "Synovitis": 0.0,
        "RelevantKeywords": "hondromalacija I stupnja, mukoidna degeneracija ACL",
    },
    {
        "SequenceID": 47,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10402431429660337753694827208437698007",
        "DetectedLanguage": "Spanish",
        "ACL": 0.0,
        "Baker's": 0.0,
        "Contusion": 0.0,
        "Effusion": 0.0,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 0.0,
        "Medial OA": 0.0,
        "PF OA": 0.3,
        "Synovitis": 0.0,
        "RelevantKeywords": "condropatia rotuliana superficial, tendinosis rotuliana",
    },
    {
        "SequenceID": 48,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10405028141768633100332386157519916334",
        "DetectedLanguage": "English",
        "ACL": 0.0,
        "Baker's": 0.0,
        "Contusion": 0.0,
        "Effusion": 0.0,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 0.0,
        "Medial OA": 0.0,
        "PF OA": 0.0,
        "Synovitis": 0.0,
        "RelevantKeywords": "minimal tendinosis extensor mechanism",
    },
    {
        "SequenceID": 49,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10407865232593182855530878478701744409",
        "DetectedLanguage": "Turkish",
        "ACL": 0.0,
        "Baker's": 0.0,
        "Contusion": 0.0,
        "Effusion": 0.2,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.3,
        "Lateral OA": 0.4,
        "MCL": 0.3,
        "Medial Meniscus": 0.8,
        "Medial OA": 0.4,
        "PF OA": 0.4,
        "Synovitis": 0.0,
        "RelevantKeywords": "bikompartmantal osteoartrit, diskoid lateral menisküs, medyal menisküs kompleks yırtık, iç yan bağ ödemi",
    },
    {
        "SequenceID": 50,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10409870090434047408497289167124673434",
        "DetectedLanguage": "English",
        "ACL": 1.0,
        "Baker's": 0.0,
        "Contusion": 0.0,
        "Effusion": 0.2,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.8,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 0.8,
        "Medial OA": 0.0,
        "PF OA": 0.0,
        "Synovitis": 0.0,
        "RelevantKeywords": "horizontal tear medial meniscus, horizontal tear lateral meniscus, complete tear ACL, minimal effusion",
    },
]

df = pd.DataFrame(data)

# Reorder columns explicitly to guarantee required structure
columns_order = [
    "SequenceID",
    "StudyInstanceUID",
    "DetectedLanguage",
    "ACL",
    "Baker's",
    "Contusion",
    "Effusion",
    "Fracture",
    "Lateral Meniscus",
    "Lateral OA",
    "MCL",
    "Medial Meniscus",
    "Medial OA",
    "PF OA",
    "Synovitis",
    "RelevantKeywords",
]

df = df[columns_order]
df.to_csv("batch_3_output.csv", index=False)
df

In [ ]:
# Output as code, batch 4

data = [
    {
        "SequenceID": 1,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10415616591430643294142401482460086713",
        "DetectedLanguage": "Bulgarian",
        "ACL": 0.8,
        "Baker's": 0.0,
        "Contusion": 0.0,
        "Effusion": 0.8,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.0,
        "MCL": 0.5,
        "Medial Meniscus": 0.5,
        "Medial OA": 0.0,
        "PF OA": 0.0,
        "Synovitis": 0.0,
        "RelevantKeywords": "subtotal anterior cruciate ligament rupture, partial medial collateral ligament tear, hemorrhagic joint effusion, medial meniscus posterior horn tear grade 2a"
    },
    {
        "SequenceID": 2,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10417042894702157309437002434676100667",
        "DetectedLanguage": "English",
        "ACL": 1.0,
        "Baker's": 0.0,
        "Contusion": 0.0,
        "Effusion": 0.3,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 1.0,
        "Medial OA": 0.0,
        "PF OA": 0.0,
        "Synovitis": 0.0,
        "RelevantKeywords": "displaced bucket-handle tear medial meniscus, anterior cruciate ligament rupture, mild joint effusion"
    },
    {
        "SequenceID": 3,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10418888172081069709796646795446885715",
        "DetectedLanguage": "English",
        "ACL": 0.2,
        "Baker's": 0.0,
        "Contusion": 0.0,
        "Effusion": 0.0,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.2,
        "MCL": 0.0,
        "Medial Meniscus": 0.0,
        "Medial OA": 0.2,
        "PF OA": 0.4,
        "Synovitis": 0.0,
        "RelevantKeywords": "patellofemoral chondrosis, semimembranosus insertional tendinosis, anterior cruciate ligament mucoid degeneration"
    },
    {
        "SequenceID": 4,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10419958821528108176389641942168280311",
        "DetectedLanguage": "Spanish",
        "ACL": 0.0,
        "Baker's": 0.0,
        "Contusion": 0.0,
        "Effusion": 0.3,
        "Fracture": 0.0,
        "Lateral Meniscus": 1.0,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 1.0,
        "Medial OA": 0.0,
        "PF OA": 0.0,
        "Synovitis": 0.0,
        "RelevantKeywords": "medial meniscus tear, lateral meniscus tear, iliotibial band friction syndrome, mild effusion"
    },
    {
        "SequenceID": 5,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10419993051271996183722192755371692029",
        "DetectedLanguage": "English",
        "ACL": 0.0,
        "Baker's": 0.3,
        "Contusion": 0.0,
        "Effusion": 0.3,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.5,
        "Lateral OA": 0.5,
        "MCL": 0.0,
        "Medial Meniscus": 0.5,
        "Medial OA": 0.5,
        "PF OA": 0.3,
        "Synovitis": 0.3,
        "RelevantKeywords": "radial tear lateral meniscus, radial tear medial meniscus, small Baker cyst, mild tricompartmental chondrosis, reactive synovitis"
    },
    {
        "SequenceID": 6,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10422130685453254363700365908336246863",
        "DetectedLanguage": "Spanish",
        "ACL": 0.0,
        "Baker's": 0.0,
        "Contusion": 0.0,
        "Effusion": 0.2,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.2,
        "MCL": 0.0,
        "Medial Meniscus": 0.2,
        "Medial OA": 0.0,
        "PF OA": 0.5,
        "Synovitis": 0.0,
        "RelevantKeywords": "patellar chondropathy, femoral trochlear chondropathy, minimal joint effusion"
    },
    {
        "SequenceID": 7,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10422536395266111565753575369125069689",
        "DetectedLanguage": "English",
        "ACL": 0.0,
        "Baker's": 0.0,
        "Contusion": 0.5,
        "Effusion": 0.5,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 0.0,
        "Medial OA": 0.0,
        "PF OA": 0.3,
        "Synovitis": 0.0,
        "RelevantKeywords": "lateral femoral condyle bone contusion, medial patellofemoral ligament sprain, moderate joint effusion, patellofemoral chondrosis"
    },
    {
        "SequenceID": 8,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10423586531262405323219596291234698626",
        "DetectedLanguage": "German",
        "ACL": 0.0,
        "Baker's": 0.0,
        "Contusion": 0.5,
        "Effusion": 0.3,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.2,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 0.8,
        "Medial OA": 0.6,
        "PF OA": 0.0,
        "Synovitis": 0.5,
        "RelevantKeywords": "complex tear medial meniscus, full thickness cartilage tear medial compartment, bone marrow edema, reactive synovitis, joint effusion"
    },
    {
        "SequenceID": 9,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10425131723456547056365404016184663035",
        "DetectedLanguage": "Spanish",
        "ACL": 0.0,
        "Baker's": 0.0,
        "Contusion": 0.0,
        "Effusion": 0.5,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.7,
        "Lateral OA": 0.5,
        "MCL": 0.0,
        "Medial Meniscus": 0.2,
        "Medial OA": 0.6,
        "PF OA": 0.9,
        "Synovitis": 0.5,
        "RelevantKeywords": "marginal tear lateral meniscus, diffuse grade 2 and 3 femorotibial chondropathy, grade 4 patellofemoral chondropathy, moderate joint effusion, diffuse synovitis"
    },
    {
        "SequenceID": 10,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10427893579790102216163565692202919399",
        "DetectedLanguage": "English",
        "ACL": 0.0,
        "Baker's": 0.0,
        "Contusion": 0.0,
        "Effusion": 0.3,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 0.0,
        "Medial OA": 0.0,
        "PF OA": 0.3,
        "Synovitis": 0.0,
        "RelevantKeywords": "grade 1 chondromalacia patella, small joint effusion"
    },
    {
        "SequenceID": 11,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10429032846672074760218668873927822056",
        "DetectedLanguage": "English",
        "ACL": 0.3,
        "Baker's": 0.0,
        "Contusion": 0.4,
        "Effusion": 0.5,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.5,
        "MCL": 0.0,
        "Medial Meniscus": 0.0,
        "Medial OA": 0.7,
        "PF OA": 0.8,
        "Synovitis": 0.6,
        "RelevantKeywords": "osteoarthritis, grade 3-4 chondromalacia trochlea, subchondral bone marrow edema, grade 1 anterior cruciate ligament injury, moderate synovitis, joint effusion"
    },
    {
        "SequenceID": 12,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10432568348356281020515782178064869805",
        "DetectedLanguage": "Bulgarian",
        "ACL": 1.0,
        "Baker's": 0.0,
        "Contusion": 0.6,
        "Effusion": 0.8,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.3,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 0.0,
        "Medial OA": 0.0,
        "PF OA": 0.0,
        "Synovitis": 0.0,
        "RelevantKeywords": "complete anterior cruciate ligament rupture, hemarthrosis joint effusion, bone marrow edema lateral femoral condyle and posterolateral tibial plateau, lateral meniscus posterior horn degeneration"
    },
    {
        "SequenceID": 13,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10434135978927658622039675788413759069",
        "DetectedLanguage": "German",
        "ACL": 0.0,
        "Baker's": 0.0,
        "Contusion": 0.7,
        "Effusion": 0.8,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.0,
        "MCL": 1.0,
        "Medial Meniscus": 0.0,
        "Medial OA": 0.0,
        "PF OA": 0.0,
        "Synovitis": 0.0,
        "RelevantKeywords": "complete posterior cruciate ligament rupture, complete proximal medial collateral ligament rupture, bone bruise, hemarthrosis"
    },
    {
        "SequenceID": 14,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10440858363080910322920496514558347561",
        "DetectedLanguage": "Croatian",
        "ACL": 1.0,
        "Baker's": 0.0,
        "Contusion": 0.6,
        "Effusion": 0.5,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.0,
        "MCL": 0.7,
        "Medial Meniscus": 0.0,
        "Medial OA": 0.0,
        "PF OA": 0.2,
        "Synovitis": 0.0,
        "RelevantKeywords": "complete anterior cruciate ligament rupture, bone edema lateral femoral condyle and lateral tibial plateau, medial collateral ligament injury, moderate joint effusion, grade 1 patellar chondromalacia"
    },
    {
        "SequenceID": 15,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10442828926117258624043497279351962738",
        "DetectedLanguage": "English",
        "ACL": 1.0,
        "Baker's": 0.0,
        "Contusion": 0.4,
        "Effusion": 0.2,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.4,
        "MCL": 0.3,
        "Medial Meniscus": 1.0,
        "Medial OA": 0.4,
        "PF OA": 0.0,
        "Synovitis": 0.0,
        "RelevantKeywords": "complete anterior cruciate ligament tear, bucket handle tear medial meniscus, bone contusion, mild medial collateral ligament sprain, fibular collateral ligament sprain, degenerative cartilage thinning"
    },
    {
        "SequenceID": 16,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10445893053733106322562086357169418409",
        "DetectedLanguage": "English",
        "ACL": 0.0,
        "Baker's": 0.3,
        "Contusion": 0.0,
        "Effusion": 0.1,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.3,
        "MCL": 0.0,
        "Medial Meniscus": 0.7,
        "Medial OA": 0.3,
        "PF OA": 0.8,
        "Synovitis": 0.0,
        "RelevantKeywords": "maceration medial meniscus body with extrusion, patellofemoral full-thickness cartilage loss, tricompartmental chondrosis, small Baker cyst"
    },
    {
        "SequenceID": 17,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10447920208680921688438738178539475509",
        "DetectedLanguage": "Croatian",
        "ACL": 0.0,
        "Baker's": 0.0,
        "Contusion": 0.0,
        "Effusion": 0.0,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.3,
        "Lateral OA": 0.5,
        "MCL": 0.0,
        "Medial Meniscus": 0.0,
        "Medial OA": 0.0,
        "PF OA": 0.4,
        "Synovitis": 0.0,
        "RelevantKeywords": "degeneration lateral meniscus anterior horn, grade 2-3 chondromalacia lateral tibial plateau, grade 2 patellar chondromalacia"
    },
    {
        "SequenceID": 18,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10454875772176439944597553400963295397",
        "DetectedLanguage": "English",
        "ACL": 0.0,
        "Baker's": 0.3,
        "Contusion": 0.0,
        "Effusion": 0.3,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 0.8,
        "Medial OA": 0.0,
        "PF OA": 0.2,
        "Synovitis": 0.0,
        "RelevantKeywords": "complex tear medial meniscus posterior horn, small popliteal cyst, small joint effusion, superficial patellar cartilage irregularity"
    },
    {
        "SequenceID": 19,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10455589411958266725060942213473374667",
        "DetectedLanguage": "English",
        "ACL": 0.0,
        "Baker's": 0.0,
        "Contusion": 0.0,
        "Effusion": 0.1,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.4,
        "MCL": 0.0,
        "Medial Meniscus": 0.8,
        "Medial OA": 0.3,
        "PF OA": 0.3,
        "Synovitis": 0.0,
        "RelevantKeywords": "oblique tear medial meniscus posterior horn with extrusion, parameniscal cyst, tricompartment chondrosis, evolving osteoarthritis"
    },
    {
        "SequenceID": 20,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10459044164648238749799867924228983674",
        "DetectedLanguage": "German",
        "ACL": 0.2,
        "Baker's": 0.0,
        "Contusion": 0.2,
        "Effusion": 0.5,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.2,
        "MCL": 0.0,
        "Medial Meniscus": 0.4,
        "Medial OA": 0.2,
        "PF OA": 0.2,
        "Synovitis": 0.5,
        "RelevantKeywords": "incipient tear medial meniscus, bone marrow edema medial tibial plateau, mucoid anterior cruciate ligament degeneration, moderate joint effusion, synovitis"
    },
    {
        "SequenceID": 21,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10459901081962705567398560859229437818",
        "DetectedLanguage": "English",
        "ACL": 1.0,
        "Baker's": 0.0,
        "Contusion": 0.4,
        "Effusion": 0.3,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.7,
        "Lateral OA": 0.0,
        "MCL": 0.5,
        "Medial Meniscus": 0.7,
        "Medial OA": 0.0,
        "PF OA": 0.0,
        "Synovitis": 0.0,
        "RelevantKeywords": "complete tear anterior cruciate ligament, old avulsion injury deep proximal medial collateral ligament, oblique vertical tear medial meniscus posterior horn, oblique vertical tear lateral meniscus posterior horn, horizontal tear lateral meniscus body, bone contusion, joint effusion"
    },
    {
        "SequenceID": 22,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10470282959537049979906878778884086884",
        "DetectedLanguage": "English",
        "ACL": 1.0,
        "Baker's": 0.0,
        "Contusion": 0.0,
        "Effusion": 0.2,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.9,
        "Lateral OA": 0.7,
        "MCL": 0.0,
        "Medial Meniscus": 0.8,
        "Medial OA": 0.0,
        "PF OA": 0.0,
        "Synovitis": 0.0,
        "RelevantKeywords": "complete tear anterior cruciate ligament, flipped vertical longitudinal tear lateral meniscus, longitudinal vertical tear medial meniscus, high grade chondral injury posterolateral tibial plateau, minimal joint effusion"
    },
    {
        "SequenceID": 23,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10472519452146922631626638652503679695",
        "DetectedLanguage": "English",
        "ACL": 0.0,
        "Baker's": 0.3,
        "Contusion": 0.0,
        "Effusion": 0.2,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 0.0,
        "Medial OA": 0.3,
        "PF OA": 0.0,
        "Synovitis": 0.0,
        "RelevantKeywords": "iliotibial band friction syndrome, healed osteochondral lesion medial femoral condyle, cartilage fissuring, ruptured Baker cyst"
    },
    {
        "SequenceID": 24,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10475397432011699698577978597417345091",
        "DetectedLanguage": "English",
        "ACL": 0.0,
        "Baker's": 0.0,
        "Contusion": 0.7,
        "Effusion": 0.4,
        "Fracture": 0.5,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.0,
        "MCL": 0.3,
        "Medial Meniscus": 0.0,
        "Medial OA": 0.0,
        "PF OA": 0.5,
        "Synovitis": 0.0,
        "RelevantKeywords": "high-grade tear medial patellofemoral ligament, osteochondral impaction fracture medial patella and lateral femoral condyle, bone contusions, low-grade medial collateral ligament sprain, chondromalacia patella, joint effusion"
    },
    {
        "SequenceID": 25,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10485026284343436524865922231113748546",
        "DetectedLanguage": "Spanish",
        "ACL": 0.0,
        "Baker's": 0.0,
        "Contusion": 0.0,
        "Effusion": 0.4,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.2,
        "MCL": 0.0,
        "Medial Meniscus": 0.8,
        "Medial OA": 0.7,
        "PF OA": 0.8,
        "Synovitis": 0.0,
        "RelevantKeywords": "complex tear medial meniscus with extrusion, grade 2 and focal grade 4 medial femorotibial chondropathy, grade 4 femoral trochlea chondropathy, mild to moderate joint effusion"
    },
    {
        "SequenceID": 26,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10491012711700179118010977263805828326",
        "DetectedLanguage": "English",
        "ACL": 0.0,
        "Baker's": 0.0,
        "Contusion": 0.0,
        "Effusion": 0.1,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.3,
        "Lateral OA": 0.3,
        "MCL": 0.0,
        "Medial Meniscus": 0.0,
        "Medial OA": 0.2,
        "PF OA": 0.8,
        "Synovitis": 0.0,
        "RelevantKeywords": "tiny partial radial tear lateral meniscus body, patellofemoral osteoarthritis, full-thickness cartilage loss patellar apex, trace joint effusion"
    },
    {
        "SequenceID": 27,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10492484093480313596330475837639172625",
        "DetectedLanguage": "English",
        "ACL": 0.0,
        "Baker's": 0.0,
        "Contusion": 0.0,
        "Effusion": 0.0,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 0.0,
        "Medial OA": 0.0,
        "PF OA": 0.0,
        "Synovitis": 0.0,
        "RelevantKeywords": "normal exam"
    },
    {
        "SequenceID": 28,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10495281520660483987888839682017074584",
        "DetectedLanguage": "English",
        "ACL": 0.3,
        "Baker's": 0.0,
        "Contusion": 0.0,
        "Effusion": 0.0,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.0,
        "MCL": 0.2,
        "Medial Meniscus": 0.8,
        "Medial OA": 0.0,
        "PF OA": 0.7,
        "Synovitis": 0.0,
        "RelevantKeywords": "complete radial tear posterior root medial meniscus with extrusion, trochlear chondrosis with high-grade cartilage loss, attenuated anterior cruciate ligament, ganglion cyst popliteus tendon sheath"
    },
    {
        "SequenceID": 29,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10495855084258410825408867079175646590",
        "DetectedLanguage": "English",
        "ACL": 0.0,
        "Baker's": 0.0,
        "Contusion": 0.6,
        "Effusion": 0.2,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.9,
        "Lateral OA": 0.8,
        "MCL": 0.0,
        "Medial Meniscus": 0.0,
        "Medial OA": 0.0,
        "PF OA": 0.8,
        "Synovitis": 0.0,
        "RelevantKeywords": "bucket handle tear lateral meniscus, high grade osteochondral defect lateral femoral condyle and tibial plateau, high grade chondral defect lateral patellar facet, subchondral bone edema, minimal joint effusion"
    },
    {
        "SequenceID": 30,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10499265498716562276605625682538326716",
        "DetectedLanguage": "German",
        "ACL": 0.0,
        "Baker's": 0.0,
        "Contusion": 0.3,
        "Effusion": 0.2,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 0.0,
        "Medial OA": 0.0,
        "PF OA": 0.4,
        "Synovitis": 0.0,
        "RelevantKeywords": "focal osteochondral lesion trochlea, subchondral bone edema, enchondroma femur metaphysis, minor joint effusion"
    },
    {
        "SequenceID": 31,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10501068619333941198079851978532746117",
        "DetectedLanguage": "Spanish",
        "ACL": 0.0,
        "Baker's": 0.0,
        "Contusion": 0.0,
        "Effusion": 0.0,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 0.0,
        "Medial OA": 0.0,
        "PF OA": 0.0,
        "Synovitis": 0.0,
        "RelevantKeywords": "Morel-Lavallee lesion anterolateral knee, mild distal quadriceps tendinosis"
    },
    {
        "SequenceID": 32,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10502440129091559437492859441158882078",
        "DetectedLanguage": "English",
        "ACL": 0.0,
        "Baker's": 0.0,
        "Contusion": 0.0,
        "Effusion": 0.0,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.3,
        "MCL": 0.0,
        "Medial Meniscus": 0.3,
        "Medial OA": 0.3,
        "PF OA": 0.0,
        "Synovitis": 0.0,
        "RelevantKeywords": "mucoid degeneration medial meniscus posterior horn, ganglion cyst adjacent to anterior cruciate ligament, incipient chondromalacia tibial compartments"
    },
    {
        "SequenceID": 33,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10505282086293876725410939800054491454",
        "DetectedLanguage": "English",
        "ACL": 0.0,
        "Baker's": 0.0,
        "Contusion": 0.0,
        "Effusion": 0.0,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 0.0,
        "Medial OA": 0.0,
        "PF OA": 0.0,
        "Synovitis": 0.0,
        "RelevantKeywords": "normal exam"
    },
    {
        "SequenceID": 34,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10511712617129216774539441890488742560",
        "DetectedLanguage": "German",
        "ACL": 0.0,
        "Baker's": 0.2,
        "Contusion": 0.0,
        "Effusion": 0.2,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.0,
        "MCL": 0.3,
        "Medial Meniscus": 0.3,
        "Medial OA": 0.0,
        "PF OA": 0.4,
        "Synovitis": 0.0,
        "RelevantKeywords": "focal deep chondral lesion medial trochlea facet, medial meniscus signal elevation, medial collateral ligament deep fibers signal changes, tiny Baker cyst"
    },
    {
        "SequenceID": 35,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10512466806929129734785318932315918522",
        "DetectedLanguage": "Croatian",
        "ACL": 0.3,
        "Baker's": 0.3,
        "Contusion": 0.5,
        "Effusion": 0.3,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.2,
        "Lateral OA": 0.6,
        "MCL": 0.0,
        "Medial Meniscus": 0.0,
        "Medial OA": 0.0,
        "PF OA": 0.6,
        "Synovitis": 0.0,
        "RelevantKeywords": "grade 3 chondromalacia patella and lateral tibial plateau, bone edema dorsal tibial plateau, mucoid degeneration cruciate ligaments, anterior cruciate ligament ganglion cyst, small Baker cyst, mild joint effusion"
    },
    {
        "SequenceID": 36,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10519480654942067141173172019311858213",
        "DetectedLanguage": "Croatian",
        "ACL": 0.0,
        "Baker's": 0.0,
        "Contusion": 0.0,
        "Effusion": 0.0,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.6,
        "MCL": 0.0,
        "Medial Meniscus": 0.0,
        "Medial OA": 0.0,
        "PF OA": 0.0,
        "Synovitis": 0.0,
        "RelevantKeywords": "grade 3 chondromalacia lateral femoral condyle"
    },
    {
        "SequenceID": 37,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10519566075312736546346469231439794627",
        "DetectedLanguage": "English",
        "ACL": 0.0,
        "Baker's": 0.0,
        "Contusion": 0.0,
        "Effusion": 0.0,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 0.0,
        "Medial OA": 0.0,
        "PF OA": 0.0,
        "Synovitis": 0.0,
        "RelevantKeywords": "normal exam"
    },
    {
        "SequenceID": 38,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10521372086358356219840179458708706015",
        "DetectedLanguage": "German",
        "ACL": 0.3,
        "Baker's": 0.7,
        "Contusion": 0.3,
        "Effusion": 0.3,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.7,
        "MCL": 0.0,
        "Medial Meniscus": 0.8,
        "Medial OA": 0.8,
        "PF OA": 0.8,
        "Synovitis": 0.0,
        "RelevantKeywords": "panosteoarthritis, full thickness chondropathy medial compartment, complex tear medial meniscus with displaced flap, mucoid anterior cruciate ligament degeneration, large lobulated Baker cyst, bone marrow edema"
    },
    {
        "SequenceID": 39,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10522585237725235866013737575867341946",
        "DetectedLanguage": "Turkish",
        "ACL": 0.3,
        "Baker's": 0.5,
        "Contusion": 0.5,
        "Effusion": 0.7,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.7,
        "Lateral OA": 0.6,
        "MCL": 0.3,
        "Medial Meniscus": 0.8,
        "Medial OA": 0.6,
        "PF OA": 0.7,
        "Synovitis": 0.0,
        "RelevantKeywords": "grade 3 degeneration and horizontal tear medial meniscus with extrusion, maceration and horizontal tear lateral meniscus, osteochondral defect lateral facet, pan-articular osteoarthritis, Baker cyst, joint effusion"
    },
    {
        "SequenceID": 40,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10524164196949381411165626176732559196",
        "DetectedLanguage": "Greek",
        "ACL": 0.0,
        "Baker's": 0.8,
        "Contusion": 0.3,
        "Effusion": 0.6,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.8,
        "MCL": 0.0,
        "Medial Meniscus": 0.7,
        "Medial OA": 0.3,
        "PF OA": 0.8,
        "Synovitis": 0.0,
        "RelevantKeywords": "degenerative osteoarthritis lateral compartment grade 3, degenerative meniscus tear, grade 4 patellofemoral chondropathy, large Baker cyst, joint effusion, prepatellar bursitis"
    },
    {
        "SequenceID": 41,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10530226787566708645477257565528473018",
        "DetectedLanguage": "German",
        "ACL": 0.0,
        "Baker's": 0.0,
        "Contusion": 0.0,
        "Effusion": 0.0,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.2,
        "MCL": 0.0,
        "Medial Meniscus": 0.0,
        "Medial OA": 0.3,
        "PF OA": 0.0,
        "Synovitis": 0.0,
        "RelevantKeywords": "grade 2 cartilage irregularities medial femorotibial compartment"
    },
    {
        "SequenceID": 42,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10532708800468111850607752497432656694",
        "DetectedLanguage": "English",
        "ACL": 0.0,
        "Baker's": 0.1,
        "Contusion": 0.0,
        "Effusion": 0.0,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 0.8,
        "Medial OA": 0.0,
        "PF OA": 0.3,
        "Synovitis": 0.0,
        "RelevantKeywords": "complex tear medial meniscus posterior horn with displaced fragment, mild patellofemoral chondrosis, trace Baker cyst"
    },
    {
        "SequenceID": 43,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10533568124272062696642461303295694739",
        "DetectedLanguage": "Spanish",
        "ACL": 0.0,
        "Baker's": 0.0,
        "Contusion": 0.0,
        "Effusion": 0.0,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 0.0,
        "Medial OA": 0.0,
        "PF OA": 0.0,
        "Synovitis": 0.0,
        "RelevantKeywords": "proximal and distal patellar tendinosis, iliotibial band friction syndrome"
    },
    {
        "SequenceID": 44,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10534260101860670685224894870019533364",
        "DetectedLanguage": "Bulgarian",
        "ACL": 0.0,
        "Baker's": 0.0,
        "Contusion": 0.0,
        "Effusion": 0.5,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 0.4,
        "Medial OA": 0.0,
        "PF OA": 0.0,
        "Synovitis": 0.0,
        "RelevantKeywords": "medial meniscus posterior horn grade 2a partial injury, joint effusion"
    },
    {
        "SequenceID": 45,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10536557774352716497356905292652597953",
        "DetectedLanguage": "English",
        "ACL": 1.0,
        "Baker's": 0.0,
        "Contusion": 0.3,
        "Effusion": 0.3,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 0.7,
        "Medial OA": 0.0,
        "PF OA": 0.3,
        "Synovitis": 0.0,
        "RelevantKeywords": "anterior cruciate ligament rupture, type 1 medial meniscus ramp lesion, minor bone contusions, grade 1 patellar chondromalacia, mild effusion"
    },
    {
        "SequenceID": 46,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10537503354035115750949427337500909327",
        "DetectedLanguage": "English",
        "ACL": 0.8,
        "Baker's": 0.0,
        "Contusion": 0.0,
        "Effusion": 0.0,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.8,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 0.8,
        "Medial OA": 0.0,
        "PF OA": 0.0,
        "Synovitis": 0.0,
        "RelevantKeywords": "high-grade anterior cruciate ligament tear, complex horizontal and oblique vertical tear medial meniscus, complex tear lateral meniscus with meniscal cyst"
    },
    {
        "SequenceID": 47,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10538502578022388392340022120166213828",
        "DetectedLanguage": "Spanish",
        "ACL": 0.0,
        "Baker's": 0.0,
        "Contusion": 0.0,
        "Effusion": 0.4,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 0.0,
        "Medial OA": 0.0,
        "PF OA": 0.4,
        "Synovitis": 0.0,
        "RelevantKeywords": "Hoffa fat pad impingement, patellar chondropathy, joint effusion"
    },
    {
        "SequenceID": 48,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10540882690103231159064894952851205573",
        "DetectedLanguage": "Spanish",
        "ACL": 0.0,
        "Baker's": 0.0,
        "Contusion": 0.0,
        "Effusion": 0.0,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 0.0,
        "Medial OA": 0.0,
        "PF OA": 0.0,
        "Synovitis": 0.0,
        "RelevantKeywords": "Hoffa fat pad impingement"
    },
    {
        "SequenceID": 49,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10540980111087702906123811593906850156",
        "DetectedLanguage": "Greek",
        "ACL": 0.0,
        "Baker's": 0.0,
        "Contusion": 0.3,
        "Effusion": 0.5,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.7,
        "Lateral OA": 0.4,
        "MCL": 0.0,
        "Medial Meniscus": 0.8,
        "Medial OA": 0.8,
        "PF OA": 0.4,
        "Synovitis": 0.6,
        "RelevantKeywords": "degenerative tear and subluxation medial meniscus, grade 4 medial femorotibial chondropathy, degenerative tear lateral meniscus anterior horn, lipoma arborescens, synovitis, joint effusion"
    },
    {
        "SequenceID": 50,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10541029960718591409318399557079055927",
        "DetectedLanguage": "English",
        "ACL": 0.5,
        "Baker's": 0.0,
        "Contusion": 0.8,
        "Effusion": 0.6,
        "Fracture": 0.8,
        "Lateral Meniscus": 0.7,
        "Lateral OA": 0.0,
        "MCL": 0.5,
        "Medial Meniscus": 0.0,
        "Medial OA": 0.0,
        "PF OA": 0.0,
        "Synovitis": 0.0,
        "RelevantKeywords": "tibial plateau split fracture, distal femur and proximal tibia bone contusion, severe partial tear posterior cruciate ligament, partial tear anterior cruciate ligament, partial tear medial collateral ligament, tear lateral meniscus anterior horn, traumatic arthritis"
    }
]

# Construct DataFrame
df = pd.DataFrame(data)

# Reorder columns as mandated by strict directives
target_columns = [
    'SequenceID',
    'StudyInstanceUID',
    'DetectedLanguage',
    'ACL',
    "Baker's",
    'Contusion',
    'Effusion',
    'Fracture',
    'Lateral Meniscus',
    'Lateral OA',
    'MCL',
    'Medial Meniscus',
    'Medial OA',
    'PF OA',
    'Synovitis',
    'RelevantKeywords'
]

df = df[target_columns]

# Export to CSV without index
df.to_csv('batch_4_output.csv', index=False)
df

In [ ]:
# Output of batch 5

data = [
    {
        "SequenceID": 1,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10541215772904358938061268226270989801",
        "DetectedLanguage": "Bulgarian",
        "ACL": 0.0,
        "Baker's": 0.0,
        "Contusion": 0.0,
        "Effusion": 0.0,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 0.0,
        "Medial OA": 0.0,
        "PF OA": 0.0,
        "Synovitis": 0.0,
        "RelevantKeywords": "patellar ligament edema, patellar tendinitis, jumper's knee",
    },
    {
        "SequenceID": 2,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10541227283184832783436242773084724216",
        "DetectedLanguage": "Turkish",
        "ACL": 0.3,
        "Baker's": 0.8,
        "Contusion": 0.6,
        "Effusion": 0.8,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.8,
        "Lateral OA": 0.6,
        "MCL": 0.3,
        "Medial Meniscus": 0.8,
        "Medial OA": 0.6,
        "PF OA": 0.0,
        "Synovitis": 0.8,
        "RelevantKeywords": "medial meniscus tear, lateral meniscus complex tear, ACL myxoid degeneration, LCL grade II sprain, MCL grade I sprain, joint effusion, bone marrow edema, osteophytes, degenerative arthritis, Baker's cyst, prepatellar edema, ganglion cyst",
    },
    {
        "SequenceID": 3,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10542526906255526405613760733936070251",
        "DetectedLanguage": "German",
        "ACL": 0.0,
        "Baker's": 0.0,
        "Contusion": 0.2,
        "Effusion": 0.0,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.7,
        "MCL": 0.0,
        "Medial Meniscus": 0.0,
        "Medial OA": 0.3,
        "PF OA": 0.0,
        "Synovitis": 0.0,
        "RelevantKeywords": "medial femoral condyle Outerbridge grade 2 chondromalacia, bone edema, anterolateral femoral condyle cysts, subchondral cysts, focal cartilage tear",
    },
    {
        "SequenceID": 4,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10546313646515875953943156661172963953",
        "DetectedLanguage": "English",
        "ACL": 0.0,
        "Baker's": 0.0,
        "Contusion": 0.0,
        "Effusion": 0.0,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 0.0,
        "Medial OA": 0.0,
        "PF OA": 0.0,
        "Synovitis": 0.0,
        "RelevantKeywords": "normal exam",
    },
    {
        "SequenceID": 5,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10549360192675575195491716398555148512",
        "DetectedLanguage": "English",
        "ACL": 0.0,
        "Baker's": 0.0,
        "Contusion": 0.0,
        "Effusion": 0.3,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 0.2,
        "Medial OA": 0.0,
        "PF OA": 0.3,
        "Synovitis": 0.0,
        "RelevantKeywords": "infrapatellar mass, nodular fasciitis, TGCT, grade 2 patellar chondromalacia, grade 2 intrasubstance signal medial meniscus, mild joint effusion",
    },
    {
        "SequenceID": 6,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10562412345413239960204081248510238736",
        "DetectedLanguage": "Turkish",
        "ACL": 0.0,
        "Baker's": 0.0,
        "Contusion": 0.0,
        "Effusion": 0.3,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 0.7,
        "Medial OA": 0.0,
        "PF OA": 0.0,
        "Synovitis": 0.0,
        "RelevantKeywords": "medial meniscus horizontal tear, extruded medial meniscus, mild joint effusion",
    },
    {
        "SequenceID": 7,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10563016328082326759266280856730347260",
        "DetectedLanguage": "Bulgarian",
        "ACL": 0.0,
        "Baker's": 0.0,
        "Contusion": 0.0,
        "Effusion": 0.3,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.4,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 0.8,
        "Medial OA": 0.5,
        "PF OA": 0.0,
        "Synovitis": 0.0,
        "RelevantKeywords": "gonarthrosis grade 2, osteophytes, cartilage thinning, medial meniscus posterior horn rupture, extrusion, lateral meniscus degeneration, meniscal cyst, loose intra-articular bodies, small effusion",
    },
    {
        "SequenceID": 8,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10574706940531650593705137704922433109",
        "DetectedLanguage": "Spanish",
        "ACL": 0.0,
        "Baker's": 0.5,
        "Contusion": 0.0,
        "Effusion": 0.5,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 0.7,
        "Medial OA": 0.0,
        "PF OA": 0.5,
        "Synovitis": 0.0,
        "RelevantKeywords": "Hoffa fat pad impingement syndrome, patellofemoral OA, iliotibial band friction syndrome, effusion, Baker's cyst, medial meniscus tear",
    },
    {
        "SequenceID": 9,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10578013533179563448221888658506023084",
        "DetectedLanguage": "Croatian",
        "ACL": 0.2,
        "Baker's": 0.2,
        "Contusion": 0.0,
        "Effusion": 0.0,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.0,
        "MCL": 0.3,
        "Medial Meniscus": 0.8,
        "Medial OA": 0.7,
        "PF OA": 0.4,
        "Synovitis": 0.0,
        "RelevantKeywords": "ACL mucoid degeneration, collateral ligament enthesopathy, MCL grade I lesion, medial meniscus complex tear, ramp lesion, patellofemoral chondromalacia grade 2, patellofemoral OA, femorotibial grade 3 chondromalacia, femorotibial OA, semimembranosus bursa fluid, popliteus tendinopathy, patellar ligament enthesopathy",
    },
    {
        "SequenceID": 10,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10580260109117947709069163110100632085",
        "DetectedLanguage": "English",
        "ACL": 0.0,
        "Baker's": 0.0,
        "Contusion": 0.0,
        "Effusion": 0.0,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 0.7,
        "Medial OA": 0.0,
        "PF OA": 0.0,
        "Synovitis": 0.0,
        "RelevantKeywords": "medial meniscus horizontal tear posterior horn and body",
    },
    {
        "SequenceID": 11,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10581718331223731732815674582085184169",
        "DetectedLanguage": "English",
        "ACL": 0.0,
        "Baker's": 0.0,
        "Contusion": 0.0,
        "Effusion": 0.0,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 0.0,
        "Medial OA": 0.0,
        "PF OA": 0.3,
        "Synovitis": 0.0,
        "RelevantKeywords": "low grade patellofemoral chondrosis, lateral patellar facet chondral fissuring",
    },
    {
        "SequenceID": 12,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10587511485561679190576188107774736852",
        "DetectedLanguage": "English",
        "ACL": 0.0,
        "Baker's": 0.0,
        "Contusion": 0.0,
        "Effusion": 0.0,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 0.6,
        "Medial OA": 0.4,
        "PF OA": 0.6,
        "Synovitis": 0.0,
        "RelevantKeywords": "osteoarthritic changes, medial meniscus degenerative tear, PCL hyperintensities, lateral patellofemoral OA, advanced chondromalacia, subchondral erosions, subcortical cysts, infrapatellar bursitis, patellar tendinitis, subcutaneous inflammation",
    },
    {
        "SequenceID": 13,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.1059889055723532336864750992142411145",
        "DetectedLanguage": "Spanish",
        "ACL": 1.0,
        "Baker's": 0.0,
        "Contusion": 0.0,
        "Effusion": 0.6,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 0.0,
        "Medial OA": 0.0,
        "PF OA": 0.0,
        "Synovitis": 0.0,
        "RelevantKeywords": "complete ACL tear, joint effusion",
    },
    {
        "SequenceID": 14,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10590148272001480024086117269799629816",
        "DetectedLanguage": "Turkish",
        "ACL": 0.0,
        "Baker's": 0.0,
        "Contusion": 0.0,
        "Effusion": 0.0,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 0.0,
        "Medial OA": 0.0,
        "PF OA": 0.0,
        "Synovitis": 0.0,
        "RelevantKeywords": "normal exam",
    },
    {
        "SequenceID": 15,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10590169032156570386146081157598789893",
        "DetectedLanguage": "English",
        "ACL": 0.0,
        "Baker's": 0.0,
        "Contusion": 0.0,
        "Effusion": 0.9,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 0.0,
        "Medial OA": 0.0,
        "PF OA": 0.2,
        "Synovitis": 0.8,
        "RelevantKeywords": "patellar lateral subluxation, medial patellar retinaculum injury grade 2, massive joint effusion, synovitis, suprapatellar bursitis, high riding patella, grade 1 patellar chondromalacia, popliteus myositis",
    },
    {
        "SequenceID": 16,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10591507262178959553431147886647585916",
        "DetectedLanguage": "Spanish",
        "ACL": 0.0,
        "Baker's": 0.5,
        "Contusion": 0.5,
        "Effusion": 0.3,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 0.8,
        "Medial OA": 0.4,
        "PF OA": 0.4,
        "Synovitis": 0.3,
        "RelevantKeywords": "complex medial meniscus tear, displaced meniscal segment, meniscal extrusion, bone marrow edema tibial plateau, femorotibial grade 2 chondropathy, grade 2 patellar chondropathy, mild effusion, synovitis, popliteal cyst with leak, patellar and quadriceps tendinosis",
    },
    {
        "SequenceID": 17,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10593694750925047629619718537436931628",
        "DetectedLanguage": "Spanish",
        "ACL": 0.5,
        "Baker's": 0.0,
        "Contusion": 0.0,
        "Effusion": 0.0,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 0.0,
        "Medial OA": 0.0,
        "PF OA": 0.0,
        "Synovitis": 0.0,
        "RelevantKeywords": "partial ACL tear",
    },
    {
        "SequenceID": 18,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10599128554628698518723223547497007137",
        "DetectedLanguage": "Turkish",
        "ACL": 0.0,
        "Baker's": 0.0,
        "Contusion": 0.0,
        "Effusion": 0.0,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 0.0,
        "Medial OA": 0.2,
        "PF OA": 0.2,
        "Synovitis": 0.0,
        "RelevantKeywords": "mild patellofemoral degeneration, mild medial femorotibial degeneration",
    },
    {
        "SequenceID": 19,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10600654987896381983520458496207741798",
        "DetectedLanguage": "Turkish",
        "ACL": 0.0,
        "Baker's": 0.8,
        "Contusion": 0.0,
        "Effusion": 0.5,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.2,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 0.8,
        "Medial OA": 0.6,
        "PF OA": 0.0,
        "Synovitis": 0.0,
        "RelevantKeywords": "leaking Baker cyst, moderate joint effusion, medial meniscus body-posterior horn degeneration, tears, horizontal cleavage tear, displaced flap tear, lateral meniscus degeneration, medial tibiofemoral full-thickness chondral fissure",
    },
    {
        "SequenceID": 20,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10601062881015669567737602031338578146",
        "DetectedLanguage": "English",
        "ACL": 0.0,
        "Baker's": 0.0,
        "Contusion": 0.0,
        "Effusion": 0.0,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.2,
        "Lateral OA": 0.4,
        "MCL": 0.0,
        "Medial Meniscus": 0.2,
        "Medial OA": 0.4,
        "PF OA": 0.4,
        "Synovitis": 0.0,
        "RelevantKeywords": "medial meniscus mucoid degeneration, lateral meniscus mucoid degeneration, tricompartmental osteoarthritis",
    },
    {
        "SequenceID": 21,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10601711635138681269270505152909539290",
        "DetectedLanguage": "Greek",
        "ACL": 0.0,
        "Baker's": 0.4,
        "Contusion": 0.0,
        "Effusion": 0.0,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.7,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 0.0,
        "Medial OA": 0.0,
        "PF OA": 0.0,
        "Synovitis": 0.0,
        "RelevantKeywords": "small Baker's cyst, lateral meniscus horizontal tear",
    },
    {
        "SequenceID": 22,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10601812195848580582490508608655011196",
        "DetectedLanguage": "English",
        "ACL": 0.0,
        "Baker's": 0.7,
        "Contusion": 0.0,
        "Effusion": 0.8,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.5,
        "MCL": 0.0,
        "Medial Meniscus": 0.7,
        "Medial OA": 0.5,
        "PF OA": 0.5,
        "Synovitis": 0.0,
        "RelevantKeywords": "medial meniscus tear, tricompartmental osteoarthritis, Baker's cyst, large joint effusion",
    },
    {
        "SequenceID": 23,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10603952065748622499432940419853732241",
        "DetectedLanguage": "Dutch",
        "ACL": 0.0,
        "Baker's": 0.8,
        "Contusion": 0.3,
        "Effusion": 0.5,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 0.0,
        "Medial OA": 0.0,
        "PF OA": 0.6,
        "Synovitis": 0.0,
        "RelevantKeywords": "patellar crista cartilage fissures, patella bone edema, suprapatellar recess distension, volumous Baker's cyst, ruptured Baker's cyst, grade 3 femoropatellar cartilage degeneration, hydrops",
    },
    {
        "SequenceID": 24,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10614508096225944703643482200835890514",
        "DetectedLanguage": "Spanish",
        "ACL": 0.0,
        "Baker's": 0.0,
        "Contusion": 0.0,
        "Effusion": 0.5,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 0.7,
        "Medial OA": 0.5,
        "PF OA": 0.5,
        "Synovitis": 0.5,
        "RelevantKeywords": "medial meniscus tear, medial femorotibial OA, patellofemoral OA, effusion, synovitis",
    },
    {
        "SequenceID": 25,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10615816709018712955642546871639043403",
        "DetectedLanguage": "Turkish",
        "ACL": 0.3,
        "Baker's": 0.0,
        "Contusion": 0.0,
        "Effusion": 0.0,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 0.2,
        "Medial OA": 0.0,
        "PF OA": 0.0,
        "Synovitis": 0.0,
        "RelevantKeywords": "medial meniscus grade 1 degeneration, ACL myxoid degeneration",
    },
    {
        "SequenceID": 26,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10619486897279334206912764495067756449",
        "DetectedLanguage": "Spanish",
        "ACL": 0.0,
        "Baker's": 0.0,
        "Contusion": 0.0,
        "Effusion": 0.2,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 0.7,
        "Medial OA": 0.0,
        "PF OA": 0.0,
        "Synovitis": 0.0,
        "RelevantKeywords": "medial meniscus horizontal oblique tear, soft tissue edema, minimal effusion, quadriceps and patellar tendinosis",
    },
    {
        "SequenceID": 27,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10619967804774235096151367674093177430",
        "DetectedLanguage": "Turkish",
        "ACL": 0.0,
        "Baker's": 0.0,
        "Contusion": 0.0,
        "Effusion": 0.6,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.7,
        "Lateral OA": 0.4,
        "MCL": 0.0,
        "Medial Meniscus": 0.7,
        "Medial OA": 0.4,
        "PF OA": 0.0,
        "Synovitis": 0.0,
        "RelevantKeywords": "subchondral cysts, hyaline cartilage thinning, patellar tendinosis, lateral meniscus tear, medial meniscus tear, joint effusion, suprapatellar effusion, prepatellar bursitis",
    },
    {
        "SequenceID": 28,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10624417936475721846835761375334601839",
        "DetectedLanguage": "Bulgarian",
        "ACL": 0.0,
        "Baker's": 0.0,
        "Contusion": 0.0,
        "Effusion": 0.0,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 0.0,
        "Medial OA": 0.0,
        "PF OA": 0.0,
        "Synovitis": 0.0,
        "RelevantKeywords": "bone islands, patellar tendinosis, jumper's knee",
    },
    {
        "SequenceID": 29,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10626071553432761206594400919097854593",
        "DetectedLanguage": "Greek",
        "ACL": 0.0,
        "Baker's": 0.0,
        "Contusion": 0.3,
        "Effusion": 0.6,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 0.0,
        "Medial OA": 0.0,
        "PF OA": 0.8,
        "Synovitis": 0.0,
        "RelevantKeywords": "tibial epiphyseal cystic lesions, bone edema, patellar cartilage degeneration ICRS grade 4, patellar subluxation, patellar tendon impingement, joint effusion, intra-articular loose bodies, patellofemoral osteoarthritis, chondromatosis",
    },
    {
        "SequenceID": 30,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10626411974489938664325062396760010912",
        "DetectedLanguage": "Bulgarian",
        "ACL": 0.0,
        "Baker's": 0.0,
        "Contusion": 0.0,
        "Effusion": 0.2,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 0.3,
        "Medial OA": 0.0,
        "PF OA": 0.0,
        "Synovitis": 0.0,
        "RelevantKeywords": "minimal joint effusion, patellar ligament inflammation, jumper's knee, medial meniscus degeneration, partial MPFL tear",
    },
    {
        "SequenceID": 31,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10627019153481144392726933328057818971",
        "DetectedLanguage": "Bulgarian",
        "ACL": 0.0,
        "Baker's": 0.0,
        "Contusion": 0.0,
        "Effusion": 0.0,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 0.3,
        "Medial OA": 0.0,
        "PF OA": 0.0,
        "Synovitis": 0.0,
        "RelevantKeywords": "medial meniscus posterior horn degenerative changes",
    },
    {
        "SequenceID": 32,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10627795990821506536342369227971125477",
        "DetectedLanguage": "Spanish",
        "ACL": 0.0,
        "Baker's": 0.0,
        "Contusion": 0.0,
        "Effusion": 0.4,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 0.3,
        "Medial OA": 0.0,
        "PF OA": 0.5,
        "Synovitis": 0.3,
        "RelevantKeywords": "medial meniscus meniscocapsular signal increase, patellar chondropathy grade 2-3, joint effusion, diffuse synovitis, quadriceps and patellar tendinosis, iliotibial band friction",
    },
    {
        "SequenceID": 33,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10631238417904652761254062374051500506",
        "DetectedLanguage": "Croatian",
        "ACL": 0.4,
        "Baker's": 0.0,
        "Contusion": 0.5,
        "Effusion": 0.0,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.0,
        "MCL": 0.2,
        "Medial Meniscus": 0.8,
        "Medial OA": 0.2,
        "PF OA": 0.0,
        "Synovitis": 0.0,
        "RelevantKeywords": "medial tibial plateau bone edema, tibial/femoral grade 1 chondromalacia, partial ACL tear, medial meniscus horizontal and radial tear, parameniscal cysts, ramp lesion, collateral ligament injury",
    },
    {
        "SequenceID": 34,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10632177122947017620482277062226288392",
        "DetectedLanguage": "Croatian",
        "ACL": 0.0,
        "Baker's": 0.0,
        "Contusion": 0.0,
        "Effusion": 0.3,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.5,
        "MCL": 0.0,
        "Medial Meniscus": 0.7,
        "Medial OA": 0.5,
        "PF OA": 0.0,
        "Synovitis": 0.0,
        "RelevantKeywords": "femorotibial chondromalacia grade III, cartilage thinning, ACL intraosseous cysts, medial meniscus horizontal tear, meniscal extrusion, ramp lesion, LCL enthesopathy, popliteus enthesopathy, quadriceps enthesopathy, patellar enthesopathy, small joint effusion",
    },
    {
        "SequenceID": 35,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10642286343421893916348249679230632066",
        "DetectedLanguage": "Turkish",
        "ACL": 0.3,
        "Baker's": 0.0,
        "Contusion": 0.5,
        "Effusion": 0.3,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.0,
        "MCL": 0.3,
        "Medial Meniscus": 0.7,
        "Medial OA": 0.7,
        "PF OA": 0.4,
        "Synovitis": 0.5,
        "RelevantKeywords": "gonarthrosis, medial meniscus degeneration and radial tear, ACL myxoid degeneration, MCL grade I sprain, suprapatellar bursitis, synovitis, patellar grade II chondromalacia, medial joint space narrowing, subchondral sclerosis and edema, popliteal ganglion cyst",
    },
    {
        "SequenceID": 36,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10648346793256473462304051626410900340",
        "DetectedLanguage": "Spanish",
        "ACL": 0.0,
        "Baker's": 0.6,
        "Contusion": 0.0,
        "Effusion": 0.6,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.3,
        "Lateral OA": 0.6,
        "MCL": 0.0,
        "Medial Meniscus": 0.0,
        "Medial OA": 0.6,
        "PF OA": 0.8,
        "Synovitis": 0.6,
        "RelevantKeywords": "lateral meniscus marginal fibrillation, femorotibial grade 2-3 chondropathy, patellofemoral grade 3-4 chondropathy, joint effusion, diffuse synovitis, ruptured popliteal cyst, quadriceps enthesopathy",
    },
    {
        "SequenceID": 37,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10650751856837517181090589171468851374",
        "DetectedLanguage": "Spanish",
        "ACL": 0.0,
        "Baker's": 0.0,
        "Contusion": 0.0,
        "Effusion": 0.5,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.8,
        "Lateral OA": 0.8,
        "MCL": 0.0,
        "Medial Meniscus": 0.7,
        "Medial OA": 0.4,
        "PF OA": 0.8,
        "Synovitis": 0.5,
        "RelevantKeywords": "medial meniscus marginal tear, lateral meniscus complex tear, displaced meniscal flap, femorotibial grade 2 chondropathy, lateral femoral condyle grade 4 chondropathy, trochlear chondropathy, patellar grade 4 chondropathy, joint effusion, diffuse synovitis, quadriceps enthesopathy",
    },
    {
        "SequenceID": 38,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10652685418462611085839874321267337079",
        "DetectedLanguage": "Dutch",
        "ACL": 0.0,
        "Baker's": 0.0,
        "Contusion": 0.0,
        "Effusion": 0.3,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 0.0,
        "Medial OA": 0.3,
        "PF OA": 0.0,
        "Synovitis": 0.0,
        "RelevantKeywords": "medial femorotibial cartilage thinning, suprapatellar recess minimal fluid, PCL mucoid degeneration, knee joint fluid",
    },
    {
        "SequenceID": 39,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10652764425372125015319160977894047658",
        "DetectedLanguage": "English",
        "ACL": 0.0,
        "Baker's": 0.0,
        "Contusion": 0.4,
        "Effusion": 0.0,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 0.0,
        "Medial OA": 0.8,
        "PF OA": 0.8,
        "Synovitis": 0.0,
        "RelevantKeywords": "medial femoral condyle full thickness chondral fissuring, median ridge full-thickness fissuring with subchondral bone marrow edema, medial patellar facet chondral loss, patellofemoral and medial compartment chondrosis",
    },
    {
        "SequenceID": 40,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10653432159060774337155662807644609967",
        "DetectedLanguage": "Spanish",
        "ACL": 0.0,
        "Baker's": 0.0,
        "Contusion": 0.0,
        "Effusion": 0.3,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.8,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 0.3,
        "Medial OA": 0.0,
        "PF OA": 0.0,
        "Synovitis": 0.0,
        "RelevantKeywords": "medial meniscus degeneration, lateral meniscus tear, lateral meniscus extrusion, displaced meniscal fragment, perimeniscal soft tissue edema, mild joint effusion",
    },
    {
        "SequenceID": 41,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10656388982978239967297620214873636996",
        "DetectedLanguage": "Bulgarian",
        "ACL": 1.0,
        "Baker's": 0.0,
        "Contusion": 0.0,
        "Effusion": 0.2,
        "Fracture": 0.0,
        "Lateral Meniscus": 1.0,
        "Lateral OA": 0.9,
        "MCL": 0.0,
        "Medial Meniscus": 0.3,
        "Medial OA": 0.0,
        "PF OA": 0.3,
        "Synovitis": 0.0,
        "RelevantKeywords": "gonarthrosis, cartilage loss, tibial degenerative cyst, osteophytes, subchondral osteosclerosis, complete lateral meniscus rupture and expulsion, meniscal cyst, medial meniscus degeneration, complete ACL rupture, patellar osteophyte",
    },
    {
        "SequenceID": 42,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10657506540109137829814058249895525831",
        "DetectedLanguage": "English",
        "ACL": 0.8,
        "Baker's": 0.0,
        "Contusion": 0.5,
        "Effusion": 0.5,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.3,
        "Lateral OA": 0.0,
        "MCL": 0.5,
        "Medial Meniscus": 0.7,
        "Medial OA": 0.0,
        "PF OA": 0.0,
        "Synovitis": 0.0,
        "RelevantKeywords": "high grade partial ACL tear, low grade partial PCL tear, partial MCL tear, partial LCL tear, popliteus tear, medial meniscus horizontal tear and contusion, lateral meniscus contusion, joint effusion",
    },
    {
        "SequenceID": 43,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10657584461509809047185519742966035356",
        "DetectedLanguage": "Croatian",
        "ACL": 0.0,
        "Baker's": 0.0,
        "Contusion": 0.0,
        "Effusion": 0.3,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.2,
        "MCL": 0.0,
        "Medial Meniscus": 0.7,
        "Medial OA": 0.0,
        "PF OA": 0.0,
        "Synovitis": 0.0,
        "RelevantKeywords": "quadriceps tendinopathy, PCL thickening, medial meniscus posterior horn horizontal tear, incipient lateral OA, suprapatellar bursa fluid",
    },
    {
        "SequenceID": 44,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10659781998801845781246073928620795082",
        "DetectedLanguage": "English",
        "ACL": 0.3,
        "Baker's": 0.7,
        "Contusion": 0.0,
        "Effusion": 0.2,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.5,
        "Lateral OA": 0.6,
        "MCL": 0.8,
        "Medial Meniscus": 0.7,
        "Medial OA": 0.8,
        "PF OA": 0.6,
        "Synovitis": 0.0,
        "RelevantKeywords": "Baker's cyst, ACL mucoid degeneration, tibial subchondral bone cyst, PCL partial tear, complete deep MCL tear, partial superficial MCL tear, medial meniscus tear and extrusion, lateral meniscus root injury, popliteus partial tear, minimal effusion, tricompartment OA, chondrosis, osteophytes",
    },
    {
        "SequenceID": 45,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10661650555947788912511478165000737040",
        "DetectedLanguage": "Croatian",
        "ACL": 0.3,
        "Baker's": 0.6,
        "Contusion": 0.5,
        "Effusion": 0.0,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.6,
        "MCL": 0.0,
        "Medial Meniscus": 0.9,
        "Medial OA": 0.9,
        "PF OA": 0.4,
        "Synovitis": 0.0,
        "RelevantKeywords": "ACL mucoid degeneration, femorotibial and PF osteophytes, medial meniscus posterior root complex tear with extrusion and flap, femorotibial grade IV chondromalacia, medial femoral condyle and tibial plateau bone edema, lateral femorotibial grade III chondromalacia, femoropatellar grade II chondromalacia, Baker's cyst, lateral meniscus ganglion cyst",
    },
    {
        "SequenceID": 46,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10673300644364559354643106316993112731",
        "DetectedLanguage": "Croatian",
        "ACL": 0.2,
        "Baker's": 0.0,
        "Contusion": 0.3,
        "Effusion": 0.3,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.3,
        "Lateral OA": 0.2,
        "MCL": 0.0,
        "Medial Meniscus": 0.0,
        "Medial OA": 0.2,
        "PF OA": 0.7,
        "Synovitis": 0.0,
        "RelevantKeywords": "patella alta, partial quadriceps tendon tear, effusion, suprapatellar fat pad edema, prepatellar subcutaneous edema, lateral patellar retinaculum distension, patellar bone edema, femorotibial grade I chondromalacia and osteophytes, patellofemoral OA, patellar cartilage denudation, trochlear grade III chondromalacia, ACL degeneration, lateral meniscus degeneration, LCL enthesopathy, popliteus bursitis, semimembranosus enthesopathy and bursitis",
    },
    {
        "SequenceID": 47,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10674673641872871321890995010616205087",
        "DetectedLanguage": "Dutch",
        "ACL": 0.0,
        "Baker's": 0.0,
        "Contusion": 0.0,
        "Effusion": 0.8,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.7,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 0.7,
        "Medial OA": 0.0,
        "PF OA": 0.0,
        "Synovitis": 0.0,
        "RelevantKeywords": "medial meniscus horizontal degenerative tear, lateral meniscus horizontal degenerative tear, hydrops, suprapatellar recess distension",
    },
    {
        "SequenceID": 48,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10675139840690240606562530926349987331",
        "DetectedLanguage": "English",
        "ACL": 0.0,
        "Baker's": 0.0,
        "Contusion": 0.6,
        "Effusion": 0.4,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 0.0,
        "Medial OA": 0.0,
        "PF OA": 0.6,
        "Synovitis": 0.0,
        "RelevantKeywords": "lateral femoral condyle bone bruise, medial patellar facet subacute osteochondral injury, MPFL chronic complete tear, knee effusion, patellar dislocation-relocation injury",
    },
    {
        "SequenceID": 49,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10676253869668012791900504544106495589",
        "DetectedLanguage": "Bulgarian",
        "ACL": 0.0,
        "Baker's": 0.0,
        "Contusion": 0.0,
        "Effusion": 0.0,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.4,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 0.8,
        "Medial OA": 0.0,
        "PF OA": 0.0,
        "Synovitis": 0.0,
        "RelevantKeywords": "medial meniscus posterior horn grade 3 tear, lateral meniscus anterior horn degeneration and cyst, soft tissue edema",
    },
    {
        "SequenceID": 50,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10684020894033412667820478260168097071",
        "DetectedLanguage": "Bulgarian",
        "ACL": 0.0,
        "Baker's": 0.0,
        "Contusion": 0.8,
        "Effusion": 0.6,
        "Fracture": 0.8,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.3,
        "MCL": 0.0,
        "Medial Meniscus": 0.0,
        "Medial OA": 0.3,
        "PF OA": 0.9,
        "Synovitis": 0.0,
        "RelevantKeywords": "joint effusion, lateral femoral condyle subchondral insufficiency fracture (SIF), extensive bone marrow edema, cartilage thinning, osteophytes, grade II LCL partial rupture, grade IV patellar chondromalacia, patellofemoral osteoarthritis, medial gonarthrosis",
    },
]

df = pd.DataFrame(data)
df.to_csv("batch_5_output.csv", index=False)
df

In [ ]:
import pandas as pd

data = [
    {
        "SequenceID": 1,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10686407053180376367066321489809395592",
        "DetectedLanguage": "Turkish",
        "ACL": 0.5,
        "Baker's": 0.0,
        "Contusion": 0.0,
        "Effusion": 0.0,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 0.3,
        "Medial OA": 0.0,
        "PF OA": 0.0,
        "Synovitis": 0.0,
        "RelevantKeywords": "medial meniskus grade I dejenerasyon, anterior capraz bag miksoid dejenerasyon",
    },
    {
        "SequenceID": 2,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10688670034893539180306173203903350286",
        "DetectedLanguage": "German",
        "ACL": 1.0,
        "Baker's": 0.0,
        "Contusion": 1.0,
        "Effusion": 1.0,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.0,
        "MCL": 0.5,
        "Medial Meniscus": 1.0,
        "Medial OA": 0.0,
        "PF OA": 0.0,
        "Synovitis": 1.0,
        "RelevantKeywords": "bone bruise, ACL Ruptur, Gelenkerguss, MCL Partiallaesion, Innenmeniskushinterhorn Riss, Synoviale Proliferationen",
    },
    {
        "SequenceID": 3,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10699348333280906956391277892869999716",
        "DetectedLanguage": "English",
        "ACL": 0.0,
        "Baker's": 0.0,
        "Contusion": 0.0,
        "Effusion": 0.6,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 0.0,
        "Medial OA": 0.0,
        "PF OA": 0.0,
        "Synovitis": 0.0,
        "RelevantKeywords": "moderate knee effusion",
    },
    {
        "SequenceID": 4,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10702284568670902901846767166679293654",
        "DetectedLanguage": "Bulgarian",
        "ACL": 0.0,
        "Baker's": 0.0,
        "Contusion": 0.0,
        "Effusion": 0.2,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 0.5,
        "Medial OA": 0.0,
        "PF OA": 0.0,
        "Synovitis": 0.0,
        "RelevantKeywords": "дегенеративни промени в медиалния менискус 2a grade, ставен излив",
    },
    {
        "SequenceID": 5,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10704240846075023334288326143979277080",
        "DetectedLanguage": "English",
        "ACL": 0.5,
        "Baker's": 0.0,
        "Contusion": 0.0,
        "Effusion": 0.2,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 0.0,
        "Medial OA": 0.0,
        "PF OA": 0.7,
        "Synovitis": 0.0,
        "RelevantKeywords": "osteochondral injury medial patellar facet, partial tear MPFL, chronic sprain LCL, chronic partial tear ACL, minimal knee effusion",
    },
    {
        "SequenceID": 6,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10713285016170190922777298047146228811",
        "DetectedLanguage": "English",
        "ACL": 0.0,
        "Baker's": 0.0,
        "Contusion": 0.0,
        "Effusion": 0.3,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 0.0,
        "Medial OA": 0.7,
        "PF OA": 0.7,
        "Synovitis": 0.0,
        "RelevantKeywords": "medial compartment OA, patellofemoral compartment OA, mild effusion",
    },
    {
        "SequenceID": 7,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10715956446507906169485719013515169244",
        "DetectedLanguage": "Greek",
        "ACL": 0.5,
        "Baker's": 1.0,
        "Contusion": 0.0,
        "Effusion": 0.8,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 1.0,
        "Medial OA": 0.0,
        "PF OA": 0.0,
        "Synovitis": 0.0,
        "RelevantKeywords": "μερική ρήξη έσω καθεκτικού, υφορική ρήξη πρόσθιου χιαστού, εκφύλιση και ρήξη έσω μηνίσκου, ύδραρθρο-αίμαρθρο, κύστη Baker",
    },
    {
        "SequenceID": 8,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10724604223190030113012801427752979879",
        "DetectedLanguage": "Turkish",
        "ACL": 0.0,
        "Baker's": 0.0,
        "Contusion": 0.0,
        "Effusion": 0.0,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 0.0,
        "Medial OA": 0.0,
        "PF OA": 0.0,
        "Synovitis": 0.0,
        "RelevantKeywords": "Normal sinirlarda bulgular",
    },
    {
        "SequenceID": 9,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10726624821670417064567481545750543282",
        "DetectedLanguage": "German",
        "ACL": 0.0,
        "Baker's": 1.0,
        "Contusion": 1.0,
        "Effusion": 0.5,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.3,
        "Lateral OA": 0.0,
        "MCL": 0.5,
        "Medial Meniscus": 1.0,
        "Medial OA": 0.8,
        "PF OA": 0.8,
        "Synovitis": 1.0,
        "RelevantKeywords": "Chondropathie retropatellar, Bone bruise, Chondropathie mediale Femurkondyle, Gelenkerguss, synoviale Proliferationen, mucoide Degeneration Aussenmeniskus, Rissbildung Innenmeniskus, MCL Partialruptur, Baker-Zyste",
    },
    {
        "SequenceID": 10,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10728448245470003618625208276233778895",
        "DetectedLanguage": "Bulgarian",
        "ACL": 0.0,
        "Baker's": 1.0,
        "Contusion": 0.0,
        "Effusion": 0.6,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 1.0,
        "Medial OA": 0.0,
        "PF OA": 0.0,
        "Synovitis": 0.0,
        "RelevantKeywords": "руптура на заден рог на медиален мениск, ставен излив, Бекерова киста",
    },
    {
        "SequenceID": 11,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10729725525284633011179038668107963557",
        "DetectedLanguage": "English",
        "ACL": 0.0,
        "Baker's": 1.0,
        "Contusion": 0.0,
        "Effusion": 0.0,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.5,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 0.4,
        "Medial OA": 0.6,
        "PF OA": 0.0,
        "Synovitis": 0.0,
        "RelevantKeywords": "Baker's cyst, tear lateral meniscus anterior horn, grade 2 intrasubstance signal medial meniscus posterior horn, osteoarthritis medial femoral condyle, Anserine bursitis",
    },
    {
        "SequenceID": 12,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10731612202208916303411280624156291171",
        "DetectedLanguage": "Turkish",
        "ACL": 0.4,
        "Baker's": 0.0,
        "Contusion": 0.0,
        "Effusion": 0.3,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.4,
        "Lateral OA": 0.3,
        "MCL": 0.4,
        "Medial Meniscus": 0.7,
        "Medial OA": 0.8,
        "PF OA": 0.0,
        "Synovitis": 1.0,
        "RelevantKeywords": "Gonatroz, femur medial kondilde osteokondral defekt, medial meniskus grade III, lateral meniskus grade II, anterior capraz bag miksoid dejenerasyon, sinovit, MCL grade I sprain",
    },
    {
        "SequenceID": 13,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10736346409444614168571281812134757732",
        "DetectedLanguage": "English",
        "ACL": 0.0,
        "Baker's": 0.0,
        "Contusion": 0.0,
        "Effusion": 0.3,
        "Fracture": 0.0,
        "Lateral Meniscus": 1.0,
        "Lateral OA": 0.3,
        "MCL": 0.0,
        "Medial Meniscus": 0.3,
        "Medial OA": 0.3,
        "PF OA": 0.9,
        "Synovitis": 0.0,
        "RelevantKeywords": "patellofemoral cartilage loss, degenerative horizontal tear lateral meniscus, medial meniscus degeneration, knee joint effusion",
    },
    {
        "SequenceID": 14,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10738640862162795323010965765443420384",
        "DetectedLanguage": "Greek",
        "ACL": 0.0,
        "Baker's": 0.0,
        "Contusion": 0.0,
        "Effusion": 0.2,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 0.0,
        "Medial OA": 0.0,
        "PF OA": 0.0,
        "Synovitis": 0.0,
        "RelevantKeywords": "μικρή ποσότητα υγρού, οίδημα λαγονοκνημιαίας ταινίας",
    },
    {
        "SequenceID": 15,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10739653650865735867752564520952904802",
        "DetectedLanguage": "English",
        "ACL": 0.0,
        "Baker's": 0.0,
        "Contusion": 0.0,
        "Effusion": 0.6,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 1.0,
        "Medial OA": 0.0,
        "PF OA": 0.8,
        "Synovitis": 1.0,
        "RelevantKeywords": "small radial tear medial meniscus, patellofemoral chondrosis and osteophytes, partial tear popliteus tendon, knee effusion with synovitis",
    },
    {
        "SequenceID": 16,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10740857795874468664161171813136103519",
        "DetectedLanguage": "Croatian",
        "ACL": 0.3,
        "Baker's": 1.0,
        "Contusion": 0.0,
        "Effusion": 0.3,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.0,
        "MCL": 0.3,
        "Medial Meniscus": 1.0,
        "Medial OA": 0.8,
        "PF OA": 0.3,
        "Synovitis": 0.0,
        "RelevantKeywords": "Chondromalacia patellae, osteoartrotske promjene, kompleksna ruptura medijalnog meniska, Baker cista, mukoidna degeneracija ACL",
    },
    {
        "SequenceID": 17,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10748806810865802591313438958975301375",
        "DetectedLanguage": "German",
        "ACL": 0.0,
        "Baker's": 0.0,
        "Contusion": 1.0,
        "Effusion": 1.0,
        "Fracture": 1.0,
        "Lateral Meniscus": 0.4,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 0.3,
        "Medial OA": 0.6,
        "PF OA": 0.5,
        "Synovitis": 0.0,
        "RelevantKeywords": "Kniegelenkserguss, Haemarthros, bone bruise, Frakturlinie posterolateraler Femurkondylus, Gonarthrose, Retropatellararthrose, Innenmeniskus mukoide Veraenderungen, Aussenmeniskus fissuraler Einriss",
    },
    {
        "SequenceID": 18,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10750145216793948201921667945805555093",
        "DetectedLanguage": "English",
        "ACL": 0.0,
        "Baker's": 1.0,
        "Contusion": 0.6,
        "Effusion": 0.8,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.5,
        "Lateral OA": 0.0,
        "MCL": 0.5,
        "Medial Meniscus": 1.0,
        "Medial OA": 0.7,
        "PF OA": 0.0,
        "Synovitis": 1.0,
        "RelevantKeywords": "Joint effusion, synovitis, suprapatellar bursitis, Baker's cyst, tear medial meniscus, degeneration/tear lateral meniscus, osteoarthritis, MCL grade 2 injury, osteonecrosis R/O",
    },
    {
        "SequenceID": 19,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10751346650129497348457481341700957445",
        "DetectedLanguage": "Turkish",
        "ACL": 0.0,
        "Baker's": 0.0,
        "Contusion": 0.0,
        "Effusion": 0.0,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 0.0,
        "Medial OA": 0.0,
        "PF OA": 0.0,
        "Synovitis": 0.0,
        "RelevantKeywords": "Normal mrg bulgulari",
    },
    {
        "SequenceID": 20,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10751624265599812767108610284996090628",
        "DetectedLanguage": "Dutch",
        "ACL": 0.3,
        "Baker's": 0.0,
        "Contusion": 0.6,
        "Effusion": 0.9,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 1.0,
        "Medial OA": 0.9,
        "PF OA": 0.0,
        "Synovitis": 0.0,
        "RelevantKeywords": "mediaal femorotibiaal kraakbeenlijden, subchondraal oedeem, transfixerende radiaire scheur mediale meniscus, mucoide degeneratie VKB, gewrichtsuitstorting, gewrichtsmuizen",
    },
    {
        "SequenceID": 21,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10756245293186761042431460535210737327",
        "DetectedLanguage": "English",
        "ACL": 0.0,
        "Baker's": 0.0,
        "Contusion": 0.0,
        "Effusion": 0.0,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 1.0,
        "Medial OA": 0.3,
        "PF OA": 0.0,
        "Synovitis": 0.0,
        "RelevantKeywords": "complex tearing posterior horn medial meniscus, parameniscal cyst, mild fraying medial cartilage",
    },
    {
        "SequenceID": 22,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10759288702627176691392048851622294054",
        "DetectedLanguage": "Turkish",
        "ACL": 0.3,
        "Baker's": 0.0,
        "Contusion": 0.8,
        "Effusion": 0.3,
        "Fracture": 0.6,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.0,
        "MCL": 0.5,
        "Medial Meniscus": 1.0,
        "Medial OA": 0.0,
        "PF OA": 0.3,
        "Synovitis": 0.0,
        "RelevantKeywords": "MCL parsiyel yirtik, tibyal plato kemik kontuzyonu ve kompresyon kirigi, mediyal meniskus horizontal oblik yirtik, medyal patellar faset intrakondral dejenerasyon, ACL edem",
    },
    {
        "SequenceID": 23,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10760847155803463653759690525160815826",
        "DetectedLanguage": "English",
        "ACL": 0.5,
        "Baker's": 0.0,
        "Contusion": 0.6,
        "Effusion": 0.6,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.5,
        "Lateral OA": 0.6,
        "MCL": 0.0,
        "Medial Meniscus": 1.0,
        "Medial OA": 0.7,
        "PF OA": 0.5,
        "Synovitis": 1.0,
        "RelevantKeywords": "vertical and radial tear medial meniscus posterior horn, chondral injury medial compartment, lateral meniscus posterior horn root injury, PCL tear, ACL partial tear, knee effusion with synovitis",
    },
    {
        "SequenceID": 24,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10761658878615069270304502752374858610",
        "DetectedLanguage": "Greek",
        "ACL": 0.0,
        "Baker's": 0.2,
        "Contusion": 0.0,
        "Effusion": 0.5,
        "Fracture": 0.0,
        "Lateral Meniscus": 1.0,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 1.0,
        "Medial OA": 0.0,
        "PF OA": 0.0,
        "Synovitis": 0.0,
        "RelevantKeywords": "εκφυλιστική ρήξη έσω μηνίσκου, παραμηνισκικές κύστεις, οριζόντια ρήξη έξω μηνίσκου, μέτρια συλλογή υγρού, κύστη Baker",
    },
    {
        "SequenceID": 25,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10762741411077456296656623422020928762",
        "DetectedLanguage": "English",
        "ACL": 0.0,
        "Baker's": 0.0,
        "Contusion": 0.0,
        "Effusion": 0.0,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.6,
        "MCL": 0.0,
        "Medial Meniscus": 1.0,
        "Medial OA": 0.6,
        "PF OA": 0.6,
        "Synovitis": 0.0,
        "RelevantKeywords": "Medial meniscus tear, OA of all three compartments",
    },
    {
        "SequenceID": 26,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10765617676329232046176300833850901740",
        "DetectedLanguage": "Spanish",
        "ACL": 0.0,
        "Baker's": 0.0,
        "Contusion": 0.0,
        "Effusion": 0.0,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 0.0,
        "Medial OA": 0.0,
        "PF OA": 0.4,
        "Synovitis": 0.0,
        "RelevantKeywords": "pinzamiento grasa de Hoffa, Condropatia rotuliana, lesjon quistica artrosinovial vs vascular",
    },
    {
        "SequenceID": 27,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10768305343887790230989914438483009428",
        "DetectedLanguage": "Croatian",
        "ACL": 1.0,
        "Baker's": 1.0,
        "Contusion": 0.0,
        "Effusion": 0.0,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 1.0,
        "Medial OA": 0.3,
        "PF OA": 0.6,
        "Synovitis": 0.0,
        "RelevantKeywords": "kosa ruptura medijalnog meniska, ramp lezija, kompletna ruptura prednje krizne sveze, hondromalacija, Baker cista, impingement suprapatelarnog masnog jastucica",
    },
    {
        "SequenceID": 28,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10769126730281890056368854871900324448",
        "DetectedLanguage": "German",
        "ACL": 1.0,
        "Baker's": 1.0,
        "Contusion": 0.3,
        "Effusion": 0.5,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.8,
        "Lateral OA": 0.3,
        "MCL": 0.4,
        "Medial Meniscus": 1.0,
        "Medial OA": 0.5,
        "PF OA": 0.3,
        "Synovitis": 0.0,
        "RelevantKeywords": "Knochenmarksoedem, Chondropathie, Lappenriss Innenmeniskus, Substanzdefekt/Riss Aussenmeniskus, VKB-Ruptur, MCL Partiallaesion, Gelenkerguss, Baker-Zyste",
    },
    {
        "SequenceID": 29,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10770877369291591553624215762368014357",
        "DetectedLanguage": "Greek",
        "ACL": 0.4,
        "Baker's": 0.4,
        "Contusion": 0.0,
        "Effusion": 0.2,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 0.0,
        "Medial OA": 0.0,
        "PF OA": 0.0,
        "Synovitis": 0.0,
        "RelevantKeywords": "παλαιά μερική ρήξη πρόσθιου χιαστού, μικρή συλλογή υγρού, κύστη Baker",
    },
    {
        "SequenceID": 30,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10774147135673365265606645781662088745",
        "DetectedLanguage": "English",
        "ACL": 0.3,
        "Baker's": 1.0,
        "Contusion": 0.0,
        "Effusion": 0.7,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 1.0,
        "Medial OA": 0.9,
        "PF OA": 0.0,
        "Synovitis": 1.0,
        "RelevantKeywords": "osteoarthritis grade 4 chondromalacia medial compartment, tear medial meniscus, grade 1 injury ACL, synovitis, suprapatellar bursitis, Baker's cyst",
    },
    {
        "SequenceID": 31,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10777331080659372520952742261140923555",
        "DetectedLanguage": "English",
        "ACL": 0.3,
        "Baker's": 0.0,
        "Contusion": 0.0,
        "Effusion": 0.5,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 0.3,
        "Medial OA": 0.5,
        "PF OA": 0.7,
        "Synovitis": 0.0,
        "RelevantKeywords": "Osteoarthritis left knee, patellofemoral cartilage wearing, grade 2 intrasubstance signal medial meniscus posterior horn, grade 1 ACL injury, iliotibial band tendinosis, joint effusion",
    },
    {
        "SequenceID": 32,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10782963448708192069959366583555573334",
        "DetectedLanguage": "English",
        "ACL": 0.0,
        "Baker's": 0.0,
        "Contusion": 0.0,
        "Effusion": 0.2,
        "Fracture": 0.0,
        "Lateral Meniscus": 1.0,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 0.0,
        "Medial OA": 0.0,
        "PF OA": 0.0,
        "Synovitis": 0.0,
        "RelevantKeywords": "lateral meniscus horizontal tear, minimal knee effusion",
    },
    {
        "SequenceID": 33,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10783501127779906316117667790784040770",
        "DetectedLanguage": "Spanish",
        "ACL": 0.0,
        "Baker's": 1.0,
        "Contusion": 0.0,
        "Effusion": 0.7,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.6,
        "MCL": 0.0,
        "Medial Meniscus": 1.0,
        "Medial OA": 0.6,
        "PF OA": 0.6,
        "Synovitis": 1.0,
        "RelevantKeywords": "Rotura de menisco interno, OA tricompartimental, Derrame con sinovitis, Quiste de Baker con fuga",
    },
    {
        "SequenceID": 34,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10790689419760453870458170966064168002",
        "DetectedLanguage": "Turkish",
        "ACL": 0.0,
        "Baker's": 0.0,
        "Contusion": 0.0,
        "Effusion": 0.0,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 0.0,
        "Medial OA": 0.0,
        "PF OA": 0.0,
        "Synovitis": 0.0,
        "RelevantKeywords": "Normal mrg bulgulari",
    },
    {
        "SequenceID": 35,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10802653832050736637820225444196833450",
        "DetectedLanguage": "English",
        "ACL": 0.0,
        "Baker's": 0.0,
        "Contusion": 0.0,
        "Effusion": 0.0,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 0.0,
        "Medial OA": 0.0,
        "PF OA": 0.0,
        "Synovitis": 0.0,
        "RelevantKeywords": "Normal study",
    },
    {
        "SequenceID": 36,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10803283287658556463524649793955055077",
        "DetectedLanguage": "English",
        "ACL": 0.0,
        "Baker's": 0.0,
        "Contusion": 0.0,
        "Effusion": 0.0,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.3,
        "MCL": 0.0,
        "Medial Meniscus": 1.0,
        "Medial OA": 0.0,
        "PF OA": 0.0,
        "Synovitis": 0.0,
        "RelevantKeywords": "buckethandle tear medial meniscus, displaced meniscal fragment, focal low grade chondrosis posterolateral tibial plateau",
    },
    {
        "SequenceID": 37,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10808663384878038371710260386807468588",
        "DetectedLanguage": "Bulgarian",
        "ACL": 0.0,
        "Baker's": 0.0,
        "Contusion": 0.0,
        "Effusion": 0.0,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 0.5,
        "Medial OA": 0.0,
        "PF OA": 0.0,
        "Synovitis": 0.0,
        "RelevantKeywords": "увреда в задния рог на медиалния менискус 2а степен, менискусова киста, ганглион киста",
    },
    {
        "SequenceID": 38,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10812894672739619566834776054635588406",
        "DetectedLanguage": "Spanish",
        "ACL": 0.0,
        "Baker's": 0.0,
        "Contusion": 0.0,
        "Effusion": 0.0,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 1.0,
        "Medial OA": 0.0,
        "PF OA": 0.0,
        "Synovitis": 0.0,
        "RelevantKeywords": "Rotura de menisco interno, Pinzamiento grasa de Hoffa, Friccion banda iliotibial",
    },
    {
        "SequenceID": 39,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10818980124981334596808438908811623528",
        "DetectedLanguage": "Dutch",
        "ACL": 0.0,
        "Baker's": 0.0,
        "Contusion": 0.3,
        "Effusion": 0.0,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 0.0,
        "Medial OA": 0.5,
        "PF OA": 0.9,
        "Synovitis": 0.0,
        "RelevantKeywords": "kraakbeenletsel graad 4 patellair, kraakbeenfissuur graad 3 mediale femurcondyl, tendinose patellapees",
    },
    {
        "SequenceID": 40,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10824420630965162625242888916805538597",
        "DetectedLanguage": "English",
        "ACL": 0.0,
        "Baker's": 0.0,
        "Contusion": 0.0,
        "Effusion": 0.0,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 0.0,
        "Medial OA": 0.0,
        "PF OA": 0.0,
        "Synovitis": 0.0,
        "RelevantKeywords": "minimal distal quadriceps insertional tendinosis",
    },
    {
        "SequenceID": 41,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10829579914809383212986717269028170263",
        "DetectedLanguage": "Spanish",
        "ACL": 0.0,
        "Baker's": 0.0,
        "Contusion": 0.0,
        "Effusion": 0.0,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 0.0,
        "Medial OA": 0.0,
        "PF OA": 0.9,
        "Synovitis": 0.0,
        "RelevantKeywords": "condropatia focal grado 4 faceta lateral rotula, tendinosis cuadricipital, friccion banda iliotibial",
    },
    {
        "SequenceID": 42,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10830288389793651695251406408666084306",
        "DetectedLanguage": "English",
        "ACL": 0.0,
        "Baker's": 0.0,
        "Contusion": 0.4,
        "Effusion": 0.2,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.7,
        "MCL": 0.0,
        "Medial Meniscus": 0.0,
        "Medial OA": 0.0,
        "PF OA": 0.7,
        "Synovitis": 0.0,
        "RelevantKeywords": "focal high grade chondral defect posterolateral tibial plateau, focal high grade chondrosis patellar facet and trochlea, minimal knee effusion",
    },
    {
        "SequenceID": 43,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10835889400414225749339608096119636738",
        "DetectedLanguage": "English",
        "ACL": 0.0,
        "Baker's": 0.0,
        "Contusion": 0.0,
        "Effusion": 0.2,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 0.0,
        "Medial OA": 0.0,
        "PF OA": 0.0,
        "Synovitis": 0.0,
        "RelevantKeywords": "partial tear PCL, subchondral cyst/intraosseous ganglion lateral femoral condyle, tendinosis Quadriceps tendon, deep infrapatellar bursitis",
    },
    {
        "SequenceID": 44,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10839126901366636822676830560774505865",
        "DetectedLanguage": "Greek",
        "ACL": 0.0,
        "Baker's": 0.0,
        "Contusion": 0.0,
        "Effusion": 0.0,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 0.0,
        "Medial OA": 0.0,
        "PF OA": 0.0,
        "Synovitis": 0.0,
        "RelevantKeywords": "Χωρίς παθολογικά ευρήματα",
    },
    {
        "SequenceID": 45,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10842906919708856642000130208791281263",
        "DetectedLanguage": "Dutch",
        "ACL": 0.0,
        "Baker's": 1.0,
        "Contusion": 0.0,
        "Effusion": 0.0,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.0,
        "MCL": 0.3,
        "Medial Meniscus": 1.0,
        "Medial OA": 0.6,
        "PF OA": 0.6,
        "Synovitis": 0.0,
        "RelevantKeywords": "horizontale scheur mediale meniscus met flap, parameniscale cyste, kraakbeenletsels graad 2-3 mediaal femorotibiaal en patellofemoraal, osteofyten, grote bakercyste",
    },
    {
        "SequenceID": 46,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10843463954622076609489312662891432839",
        "DetectedLanguage": "English",
        "ACL": 0.0,
        "Baker's": 0.0,
        "Contusion": 0.0,
        "Effusion": 0.1,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.5,
        "MCL": 0.0,
        "Medial Meniscus": 0.0,
        "Medial OA": 0.0,
        "PF OA": 0.5,
        "Synovitis": 0.0,
        "RelevantKeywords": "pes anserine bursitis, patellofemoral chondrosis, high-grade chondrosis lateral tibial plateau, trace joint fluid",
    },
    {
        "SequenceID": 47,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10843917293035279597622029625809500972",
        "DetectedLanguage": "Croatian",
        "ACL": 0.0,
        "Baker's": 0.6,
        "Contusion": 0.0,
        "Effusion": 0.0,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.0,
        "LCL": 0.3,
        "MCL": 0.2,
        "Medial Meniscus": 1.0,
        "Medial OA": 0.4,
        "PF OA": 0.6,
        "Synovitis": 0.0,
        "RelevantKeywords": "hondromalacija, kompleksna ruptura medijalnog meniska, ramp lezija, edem LCL, Baker cista",
    },
    {
        "SequenceID": 48,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10848350333764008140459854470293737650",
        "DetectedLanguage": "Spanish",
        "ACL": 0.0,
        "Baker's": 0.0,
        "Contusion": 0.0,
        "Effusion": 0.0,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 0.0,
        "Medial OA": 0.0,
        "PF OA": 0.6,
        "Synovitis": 0.0,
        "RelevantKeywords": "OA patelofemoral moderada",
    },
    {
        "SequenceID": 49,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10850742579000782417673172155504794301",
        "DetectedLanguage": "French",
        "ACL": 0.0,
        "Baker's": 0.6,
        "Contusion": 0.0,
        "Effusion": 0.3,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.0,
        "MCL": 0.0,
        "Medial Meniscus": 0.0,
        "Medial OA": 0.0,
        "PF OA": 0.5,
        "Synovitis": 0.0,
        "RelevantKeywords": "petit epanchement intra-articulaire, kyste poplite, changements degeneratifs patello-femoraux",
    },
    {
        "SequenceID": 50,
        "StudyInstanceUID": "1.2.826.0.1.3680043.8.498.10851562687641986545442408470139368318",
        "DetectedLanguage": "English",
        "ACL": 0.0,
        "Baker's": 0.0,
        "Contusion": 0.4,
        "Effusion": 0.3,
        "Fracture": 0.0,
        "Lateral Meniscus": 0.0,
        "Lateral OA": 0.8,
        "MCL": 0.0,
        "Medial Meniscus": 0.0,
        "Medial OA": 0.0,
        "PF OA": 0.0,
        "Synovitis": 0.0,
        "RelevantKeywords": "Lateral compartment chondrosis, full-thickness defect lateral femoral condyle, subchondral cystic change, marrow edema, mild quadriceps tendinosis, small joint effusion",
    },
]

df = pd.DataFrame(data)
df = df[df_cols]
df.to_csv("batch_6_output.csv", index=False)

df

In [ ]:
# Batch 7

import pandas as pd

data = [
    {
        'SequenceID': 1,
        'StudyInstanceUID': '1.2.826.0.1.3680043.8.498.10852517092524464338210061305792376636',
        'DetectedLanguage': 'English',
        'ACL': 1.0,
        "Baker's": 0.0,
        'Contusion': 1.0,
        'Effusion': 0.7,
        'Fracture': 0.0,
        'Lateral Meniscus': 0.5,
        'Lateral OA': 0.0,
        'MCL': 0.5,
        'Medial Meniscus': 1.0,
        'Medial OA': 0.0,
        'PF OA': 0.0,
        'Synovitis': 1.0,
        'RelevantKeywords': 'ACL tear, medial meniscus tear, lateral meniscus tear, MCL sprain, effusion, synovitis, bone contusion'
    },
    {
        'SequenceID': 2,
        'StudyInstanceUID': '1.2.826.0.1.3680043.8.498.10854012268828835845581966634225522931',
        'DetectedLanguage': 'German',
        'ACL': 0.5,
        "Baker's": 0.0,
        'Contusion': 1.0,
        'Effusion': 1.0,
        'Fracture': 0.0,
        'Lateral Meniscus': 0.0,
        'Lateral OA': 0.0,
        'MCL': 0.0,
        'Medial Meniscus': 0.0,
        'Medial OA': 0.0,
        'PF OA': 0.0,
        'Synovitis': 0.0,
        'RelevantKeywords': 'Knochenmarksödem, hämorrhagischer Gelenkerguss, VKB-Läsion'
    },
    {
        'SequenceID': 3,
        'StudyInstanceUID': '1.2.826.0.1.3680043.8.498.10854223557359336882116168394037581766',
        'DetectedLanguage': 'Turkish',
        'ACL': 0.5,
        "Baker's": 0.0,
        'Contusion': 0.0,
        'Effusion': 0.3,
        'Fracture': 0.0,
        'Lateral Meniscus': 0.0,
        'Lateral OA': 0.0,
        'MCL': 0.0,
        'Medial Meniscus': 1.0,
        'Medial OA': 1.0,
        'PF OA': 0.5,
        'Synovitis': 0.0,
        'RelevantKeywords': 'medial menisküs yırtığı, ACL dejenerasyonu, kondromalazi, dejeneratif artrit, ganglion kisti, loose body'
    },
    {
        'SequenceID': 4,
        'StudyInstanceUID': '1.2.826.0.1.3680043.8.498.10855290714402117092990651362697310270',
        'DetectedLanguage': 'Turkish',
        'ACL': 0.5,
        "Baker's": 0.0,
        'Contusion': 1.0,
        'Effusion': 0.5,
        'Fracture': 0.0,
        'Lateral Meniscus': 0.5,
        'Lateral OA': 0.0,
        'MCL': 0.5,
        'Medial Meniscus': 1.0,
        'Medial OA': 0.0,
        'PF OA': 0.0,
        'Synovitis': 0.5,
        'RelevantKeywords': 'medial menisküs grade III, lateral menisküs grade II, ACL miksoid dejenerasyon, kemik iliği ödemi, suprapatellar bursit'
    },
    {
        'SequenceID': 5,
        'StudyInstanceUID': '1.2.826.0.1.3680043.8.498.10857485602186206735293765836065270655',
        'DetectedLanguage': 'Spanish',
        'ACL': 0.0,
        "Baker's": 0.0,
        'Contusion': 0.0,
        'Effusion': 0.3,
        'Fracture': 0.0,
        'Lateral Meniscus': 0.0,
        'Lateral OA': 1.0,
        'MCL': 0.0,
        'Medial Meniscus': 0.0,
        'Medial OA': 0.0,
        'PF OA': 1.0,
        'Synovitis': 1.0,
        'RelevantKeywords': 'condropatía grado 3, úlceras condrales, derrame articular, sinovitis difusa'
    },
    {
        'SequenceID': 6,
        'StudyInstanceUID': '1.2.826.0.1.3680043.8.498.10857771747813858957718058322384884551',
        'DetectedLanguage': 'Bulgarian',
        'ACL': 0.0,
        "Baker's": 0.0,
        'Contusion': 0.3,
        'Effusion': 0.0,
        'Fracture': 0.0,
        'Lateral Meniscus': 0.0,
        'Lateral OA': 0.0,
        'MCL': 0.0,
        'Medial Meniscus': 1.0,
        'Medial OA': 0.0,
        'PF OA': 0.0,
        'Synovitis': 0.0,
        'RelevantKeywords': 'лезия на медиалния менискус grade 3, киста, костномозъчен едем'
    },
    {
        'SequenceID': 7,
        'StudyInstanceUID': '1.2.826.0.1.3680043.8.498.10862481204237376580557516731952961213',
        'DetectedLanguage': 'English',
        'ACL': 0.3,
        "Baker's": 0.0,
        'Contusion': 0.0,
        'Effusion': 0.3,
        'Fracture': 0.0,
        'Lateral Meniscus': 0.0,
        'Lateral OA': 0.0,
        'MCL': 0.0,
        'Medial Meniscus': 1.0,
        'Medial OA': 0.5,
        'PF OA': 0.0,
        'Synovitis': 0.0,
        'RelevantKeywords': 'mild effusion, incomplete tearing posterior root medial meniscus, medial compartment cartilage degeneration'
    },
    {
        'SequenceID': 8,
        'StudyInstanceUID': '1.2.826.0.1.3680043.8.498.10867622488467375444427908425709367385',
        'DetectedLanguage': 'English',
        'ACL': 0.0,
        "Baker's": 0.0,
        'Contusion': 0.0,
        'Effusion': 0.0,
        'Fracture': 0.0,
        'Lateral Meniscus': 0.0,
        'Lateral OA': 0.0,
        'MCL': 0.0,
        'Medial Meniscus': 0.0,
        'Medial OA': 0.0,
        'PF OA': 0.0,
        'Synovitis': 0.0,
        'RelevantKeywords': 'Normal'
    },
    {
        'SequenceID': 9,
        'StudyInstanceUID': '1.2.826.0.1.3680043.8.498.10876177342063734507664188801203192617',
        'DetectedLanguage': 'Dutch',
        'ACL': 0.0,
        "Baker's": 0.0,
        'Contusion': 0.0,
        'Effusion': 0.3,
        'Fracture': 0.0,
        'Lateral Meniscus': 0.8,
        'Lateral OA': 0.0,
        'MCL': 0.0,
        'Medial Meniscus': 1.0,
        'Medial OA': 0.0,
        'PF OA': 0.0,
        'Synovitis': 0.0,
        'RelevantKeywords': 'artrosynoviale cyste, degeneratieve scheur laterale meniscus, degeneratieve scheur mediale meniscus, verplaatst fragment'
    },
    {
        'SequenceID': 10,
        'StudyInstanceUID': '1.2.826.0.1.3680043.8.498.10880553082640412137979450100928382021',
        'DetectedLanguage': 'Croatian',
        'ACL': 0.3,
        "Baker's": 1.0,
        'Contusion': 0.0,
        'Effusion': 0.0,
        'Fracture': 0.0,
        'Lateral Meniscus': 0.0,
        'Lateral OA': 0.5,
        'MCL': 0.5,
        'Medial Meniscus': 1.0,
        'Medial OA': 0.5,
        'PF OA': 0.5,
        'Synovitis': 0.0,
        'RelevantKeywords': 'hondromalacija, osteofiti, medijalni menisk ruptura, MCL periligamentarni edem, Poplitealna cista'
    },
    {
        'SequenceID': 11,
        'StudyInstanceUID': '1.2.826.0.1.3680043.8.498.10884987036948550523442514147204321748',
        'DetectedLanguage': 'English',
        'ACL': 0.0,
        "Baker's": 0.0,
        'Contusion': 0.0,
        'Effusion': 0.0,
        'Fracture': 0.0,
        'Lateral Meniscus': 0.0,
        'Lateral OA': 0.0,
        'MCL': 0.0,
        'Medial Meniscus': 0.0,
        'Medial OA': 0.0,
        'PF OA': 0.0,
        'Synovitis': 0.0,
        'RelevantKeywords': 'No evidence of meniscal tear or ligamentous injury'
    },
    {
        'SequenceID': 12,
        'StudyInstanceUID': '1.2.826.0.1.3680043.8.498.10886583574130697346788286647281863932',
        'DetectedLanguage': 'Spanish',
        'ACL': 0.0,
        "Baker's": 0.0,
        'Contusion': 0.0,
        'Effusion': 0.3,
        'Fracture': 0.0,
        'Lateral Meniscus': 0.0,
        'Lateral OA': 0.5,
        'MCL': 1.0,
        'Medial Meniscus': 0.8,
        'Medial OA': 0.0,
        'PF OA': 0.5,
        'Synovitis': 0.0,
        'RelevantKeywords': 'esguince ligamento colateral medial, rotura periférica menisco medial, condropatía'
    },
    {
        'SequenceID': 13,
        'StudyInstanceUID': '1.2.826.0.1.3680043.8.498.10890699078723972693302029683905114834',
        'DetectedLanguage': 'Greek',
        'ACL': 0.0,
        "Baker's": 1.0,
        'Contusion': 0.0,
        'Effusion': 0.5,
        'Fracture': 0.0,
        'Lateral Meniscus': 0.5,
        'Lateral OA': 0.5,
        'MCL': 0.0,
        'Medial Meniscus': 1.0,
        'Medial OA': 1.0,
        'PF OA': 0.5,
        'Synovitis': 1.0,
        'RelevantKeywords': 'εκφύλιση και υπεξάρθρημα μηνίσκου, οστεοαρθρίτιδα, κύστη Baker, υμενίτιδα, χονδρομάτωση'
    },
    {
        'SequenceID': 14,
        'StudyInstanceUID': '1.2.826.0.1.3680043.8.498.10895521387894821238259709507311341758',
        'DetectedLanguage': 'French',
        'ACL': 0.3,
        "Baker's": 0.0,
        'Contusion': 0.0,
        'Effusion': 0.7,
        'Fracture': 0.0,
        'Lateral Meniscus': 0.0,
        'Lateral OA': 0.3,
        'MCL': 0.8,
        'Medial Meniscus': 1.0,
        'Medial OA': 0.5,
        'PF OA': 0.8,
        'Synovitis': 0.0,
        'RelevantKeywords': 'déchirure ménisque interne, entorse LCI, bursite, chondropathie, épanchement'
    },
    {
        'SequenceID': 15,
        'StudyInstanceUID': '1.2.826.0.1.3680043.8.498.10896245499713818848442117831482755673',
        'DetectedLanguage': 'Turkish',
        'ACL': 0.8,
        "Baker's": 0.0,
        'Contusion': 0.0,
        'Effusion': 0.3,
        'Fracture': 0.0,
        'Lateral Meniscus': 0.0,
        'Lateral OA': 0.0,
        'MCL': 0.8,
        'Medial Meniscus': 1.0,
        'Medial OA': 0.0,
        'PF OA': 0.0,
        'Synovitis': 0.0,
        'RelevantKeywords': 'medial menisküs grade III ve horizontal yırtık, PCL sprain, MCL grade I sprain'
    },
    {
        'SequenceID': 16,
        'StudyInstanceUID': '1.2.826.0.1.3680043.8.498.10902166579916526176697193295886992412',
        'DetectedLanguage': 'French',
        'ACL': 0.0,
        "Baker's": 0.0,
        'Contusion': 0.0,
        'Effusion': 0.3,
        'Fracture': 0.0,
        'Lateral Meniscus': 0.0,
        'Lateral OA': 0.0,
        'MCL': 0.0,
        'Medial Meniscus': 0.0,
        'Medial OA': 0.0,
        'PF OA': 0.0,
        'Synovitis': 0.0,
        'RelevantKeywords': 'minime épanchement articulaire'
    },
    {
        'SequenceID': 17,
        'StudyInstanceUID': '1.2.826.0.1.3680043.8.498.10903003118372337016190215211789983264',
        'DetectedLanguage': 'German',
        'ACL': 0.3,
        "Baker's": 0.0,
        'Contusion': 0.3,
        'Effusion': 0.3,
        'Fracture': 0.0,
        'Lateral Meniscus': 0.3,
        'Lateral OA': 0.0,
        'MCL': 0.0,
        'Medial Meniscus': 1.0,
        'Medial OA': 1.0,
        'PF OA': 0.0,
        'Synovitis': 0.0,
        'RelevantKeywords': 'ACL mukoid, Meniskusdegeneration, horizontale Rissbildung Innenmeniskus, Chondropathie medial, Gelenkerguss'
    },
    {
        'SequenceID': 18,
        'StudyInstanceUID': '1.2.826.0.1.3680043.8.498.10917389623891555306584069980815338466',
        'DetectedLanguage': 'English',
        'ACL': 0.3,
        "Baker's": 0.5,
        'Contusion': 0.0,
        'Effusion': 0.3,
        'Fracture': 0.0,
        'Lateral Meniscus': 1.0,
        'Lateral OA': 0.0,
        'MCL': 0.0,
        'Medial Meniscus': 0.0,
        'Medial OA': 0.0,
        'PF OA': 0.0,
        'Synovitis': 0.3,
        'RelevantKeywords': 'small joint effusion, mild synovitis, ACL mucoid degeneration, lateral meniscus degeneration and fraying, tiny Baker\'s cyst'
    },
    {
        'SequenceID': 19,
        'StudyInstanceUID': '1.2.826.0.1.3680043.8.498.10917606677895195837968927716728385881',
        'DetectedLanguage': 'Dutch',
        'ACL': 0.0,
        "Baker's": 0.0,
        'Contusion': 0.0,
        'Effusion': 0.3,
        'Fracture': 1.0,
        'Lateral Meniscus': 0.0,
        'Lateral OA': 1.0,
        'MCL': 0.0,
        'Medial Meniscus': 0.0,
        'Medial OA': 0.0,
        'PF OA': 1.0,
        'Synovitis': 0.0,
        'RelevantKeywords': 'kraakbeendefect laterale femurcondyl, chondrale fractuur, kraakbeenfissuur patellair'
    },
    {
        'SequenceID': 20,
        'StudyInstanceUID': '1.2.826.0.1.3680043.8.498.10918580785296934228715514366408269433',
        'DetectedLanguage': 'Croatian',
        'ACL': 0.0,
        "Baker's": 1.0,
        'Contusion': 1.0,
        'Effusion': 0.3,
        'Fracture': 0.0,
        'Lateral Meniscus': 0.5,
        'Lateral OA': 0.5,
        'MCL': 0.5,
        'Medial Meniscus': 1.0,
        'Medial OA': 0.5,
        'PF OA': 1.0,
        'Synovitis': 0.0,
        'RelevantKeywords': 'ruptura medijalnog meniska, koštani edem, MCL distenzija, hondromalacija, Baker cista'
    },
    {
        'SequenceID': 21,
        'StudyInstanceUID': '1.2.826.0.1.3680043.8.498.10919665962088003807153039302898262235',
        'DetectedLanguage': 'Bulgarian',
        'ACL': 0.0,
        "Baker's": 0.0,
        'Contusion': 0.3,
        'Effusion': 0.3,
        'Fracture': 0.0,
        'Lateral Meniscus': 0.0,
        'Lateral OA': 0.0,
        'MCL': 0.0,
        'Medial Meniscus': 1.0,
        'Medial OA': 1.0,
        'PF OA': 1.0,
        'Synovitis': 0.0,
        'RelevantKeywords': 'гонартроза, дегенерирал медиален менискус, киста на медиалния менискус, хондромалация на пателата'
    },
    {
        'SequenceID': 22,
        'StudyInstanceUID': '1.2.826.0.1.3680043.8.498.10920269679387171659375091645187772303',
        'DetectedLanguage': 'English',
        'ACL': 0.5,
        "Baker's": 0.0,
        'Contusion': 0.0,
        'Effusion': 0.3,
        'Fracture': 0.0,
        'Lateral Meniscus': 0.0,
        'Lateral OA': 0.0,
        'MCL': 0.0,
        'Medial Meniscus': 1.0,
        'Medial OA': 0.0,
        'PF OA': 0.0,
        'Synovitis': 0.0,
        'RelevantKeywords': 'partial tear medial meniscus root, parameniscal cyst, ACL attenuation'
    },
    {
        'SequenceID': 23,
        'StudyInstanceUID': '1.2.826.0.1.3680043.8.498.10924685588962763768106617368647678902',
        'DetectedLanguage': 'French',
        'ACL': 0.0,
        "Baker's": 0.0,
        'Contusion': 0.3,
        'Effusion': 0.0,
        'Fracture': 0.0,
        'Lateral Meniscus': 0.0,
        'Lateral OA': 0.0,
        'MCL': 0.0,
        'Medial Meniscus': 0.0,
        'Medial OA': 0.0,
        'PF OA': 0.5,
        'Synovitis': 0.0,
        'RelevantKeywords': 'chondropatie rétropatellaire modérée, œdème osseux condyle médial'
    },
    {
        'SequenceID': 24,
        'StudyInstanceUID': '1.2.826.0.1.3680043.8.498.10924995713871537837233985073507509566',
        'DetectedLanguage': 'German',
        'ACL': 0.0,
        "Baker's": 0.0,
        'Contusion': 0.0,
        'Effusion': 0.3,
        'Fracture': 0.0,
        'Lateral Meniscus': 0.0,
        'Lateral OA': 0.0,
        'MCL': 0.0,
        'Medial Meniscus': 0.0,
        'Medial OA': 0.0,
        'PF OA': 0.3,
        'Synovitis': 0.0,
        'RelevantKeywords': 'Knorpelirregularität Patellafirst, akzentuierte Gelenkflüssigkeit'
    },
    {
        'SequenceID': 25,
        'StudyInstanceUID': '1.2.826.0.1.3680043.8.498.10927716750119165925963563081261605359',
        'DetectedLanguage': 'English',
        'ACL': 1.0,
        "Baker's": 0.5,
        'Contusion': 0.0,
        'Effusion': 0.3,
        'Fracture': 0.0,
        'Lateral Meniscus': 0.0,
        'Lateral OA': 0.5,
        'MCL': 0.8,
        'Medial Meniscus': 1.0,
        'Medial OA': 0.5,
        'PF OA': 0.0,
        'Synovitis': 0.0,
        'RelevantKeywords': 'completely torn ACL, PCL tear, proximal MCL sprain, FCL sprain, massive radial tear medial meniscus, chondropathy'
    },
    {
        'SequenceID': 26,
        'StudyInstanceUID': '1.2.826.0.1.3680043.8.498.10930374192038896613513882597332665403',
        'DetectedLanguage': 'English',
        'ACL': 0.0,
        "Baker's": 0.3,
        'Contusion': 0.0,
        'Effusion': 0.0,
        'Fracture': 0.0,
        'Lateral Meniscus': 0.0,
        'Lateral OA': 0.0,
        'MCL': 0.0,
        'Medial Meniscus': 0.0,
        'Medial OA': 0.0,
        'PF OA': 0.0,
        'Synovitis': 0.0,
        'RelevantKeywords': 'trace Baker\'s cyst, small parameniscal ganglion cyst'
    },
    {
        'SequenceID': 27,
        'StudyInstanceUID': '1.2.826.0.1.3680043.8.498.10931111730564885182398163970692599398',
        'DetectedLanguage': 'Croatian',
        'ACL': 0.0,
        "Baker's": 1.0,
        'Contusion': 0.0,
        'Effusion': 0.3,
        'Fracture': 0.0,
        'Lateral Meniscus': 0.0,
        'Lateral OA': 0.5,
        'MCL': 0.5,
        'Medial Meniscus': 1.0,
        'Medial OA': 0.5,
        'PF OA': 0.5,
        'Synovitis': 0.0,
        'RelevantKeywords': 'hondromalacija, medijalni menisk radijalna ruptura i ekstruzija, MCL distenzija, Baker cista'
    },
    {
        'SequenceID': 28,
        'StudyInstanceUID': '1.2.826.0.1.3680043.8.498.10932818400202426453525918723737011737',
        'DetectedLanguage': 'Turkish',
        'ACL': 1.0,
        "Baker's": 0.0,
        'Contusion': 0.3,
        'Effusion': 0.3,
        'Fracture': 0.0,
        'Lateral Meniscus': 0.0,
        'Lateral OA': 0.0,
        'MCL': 0.0,
        'Medial Meniscus': 1.0,
        'Medial OA': 0.0,
        'PF OA': 0.0,
        'Synovitis': 0.0,
        'RelevantKeywords': 'ön çapraz bağ yırtığı, tibia anteriora translasyon, medyal menisküs periferal longitudinal yırtık, kemik iliği ödemi'
    },
    {
        'SequenceID': 29,
        'StudyInstanceUID': '1.2.826.0.1.3680043.8.498.10933003998197427979325590791540457658',
        'DetectedLanguage': 'English',
        'ACL': 0.0,
        "Baker's": 0.0,
        'Contusion': 0.0,
        'Effusion': 0.0,
        'Fracture': 0.0,
        'Lateral Meniscus': 0.0,
        'Lateral OA': 0.0,
        'MCL': 0.0,
        'Medial Meniscus': 0.0,
        'Medial OA': 0.0,
        'PF OA': 0.0,
        'Synovitis': 0.0,
        'RelevantKeywords': 'tiny ganglion formation posteromedial aspect'
    },
    {
        'SequenceID': 30,
        'StudyInstanceUID': '1.2.826.0.1.3680043.8.498.10937181824204907065071760564745305722',
        'DetectedLanguage': 'English',
        'ACL': 1.0,
        "Baker's": 0.0,
        'Contusion': 1.0,
        'Effusion': 0.3,
        'Fracture': 0.5,
        'Lateral Meniscus': 0.8,
        'Lateral OA': 0.0,
        'MCL': 0.0,
        'Medial Meniscus': 0.0,
        'Medial OA': 0.0,
        'PF OA': 0.5,
        'Synovitis': 0.5,
        'RelevantKeywords': 'ACL full thickness tear, lateral meniscus radial tear, bone marrow edema, subchondral fracture, mild synovitis, patellar chondromalacia'
    },
    {
        'SequenceID': 31,
        'StudyInstanceUID': '1.2.826.0.1.3680043.8.498.10937504474673840994058935502377258768',
        'DetectedLanguage': 'Spanish',
        'ACL': 0.0,
        "Baker's": 0.0,
        'Contusion': 0.0,
        'Effusion': 0.0,
        'Fracture': 0.0,
        'Lateral Meniscus': 0.0,
        'Lateral OA': 0.0,
        'MCL': 0.0,
        'Medial Meniscus': 0.0,
        'Medial OA': 0.0,
        'PF OA': 0.0,
        'Synovitis': 0.0,
        'RelevantKeywords': 'formación seudonodular en unión miotendínea de la banda iliotibial'
    },
    {
        'SequenceID': 32,
        'StudyInstanceUID': '1.2.826.0.1.3680043.8.498.10937585701850306334458420860276069606',
        'DetectedLanguage': 'Croatian',
        'ACL': 0.5,
        "Baker's": 0.0,
        'Contusion': 0.3,
        'Effusion': 0.0,
        'Fracture': 0.0,
        'Lateral Meniscus': 0.0,
        'Lateral OA': 0.5,
        'MCL': 0.0,
        'Medial Meniscus': 0.0,
        'Medial OA': 0.5,
        'PF OA': 0.5,
        'Synovitis': 0.0,
        'RelevantKeywords': 'koštani edem medijalnog kondila, hondromalacija, ACL parcijalna ruptura'
    },
    {
        'SequenceID': 33,
        'StudyInstanceUID': '1.2.826.0.1.3680043.8.498.10938719716535758543863717421521822626',
        'DetectedLanguage': 'English',
        'ACL': 0.0,
        "Baker's": 0.0,
        'Contusion': 0.0,
        'Effusion': 0.0,
        'Fracture': 0.0,
        'Lateral Meniscus': 0.0,
        'Lateral OA': 1.0,
        'MCL': 0.0,
        'Medial Meniscus': 1.0,
        'Medial OA': 1.0,
        'PF OA': 1.0,
        'Synovitis': 0.0,
        'RelevantKeywords': 'medial meniscus tear, OA of all three compartments'
    },
    {
        'SequenceID': 34,
        'StudyInstanceUID': '1.2.826.0.1.3680043.8.498.10939964887585312209861601414669494409',
        'DetectedLanguage': 'Spanish',
        'ACL': 0.0,
        "Baker's": 1.0,
        'Contusion': 0.0,
        'Effusion': 0.3,
        'Fracture': 0.0,
        'Lateral Meniscus': 0.0,
        'Lateral OA': 1.0,
        'MCL': 0.0,
        'Medial Meniscus': 1.0,
        'Medial OA': 1.0,
        'PF OA': 1.0,
        'Synovitis': 0.0,
        'RelevantKeywords': 'rotura de menisco interno, artrosis tricompartimental, síndrome de pinzamiento Hoffa, derrame leve, quiste de Baker'
    },
    {
        'SequenceID': 35,
        'StudyInstanceUID': '1.2.826.0.1.3680043.8.498.10951328287594724738476663200798896670',
        'DetectedLanguage': 'Turkish',
        'ACL': 0.0,
        "Baker's": 0.0,
        'Contusion': 0.0,
        'Effusion': 0.0,
        'Fracture': 0.0,
        'Lateral Meniscus': 0.0,
        'Lateral OA': 0.0,
        'MCL': 0.0,
        'Medial Meniscus': 0.8,
        'Medial OA': 0.0,
        'PF OA': 0.0,
        'Synovitis': 0.0,
        'RelevantKeywords': 'medial menisküste grade 2 meniskopati'
    },
    {
        'SequenceID': 36,
        'StudyInstanceUID': '1.2.826.0.1.3680043.8.498.10959153852077371752490787490374555514',
        'DetectedLanguage': 'English',
        'ACL': 0.0,
        "Baker's": 0.0,
        'Contusion': 0.0,
        'Effusion': 0.0,
        'Fracture': 0.0,
        'Lateral Meniscus': 0.0,
        'Lateral OA': 0.0,
        'MCL': 0.0,
        'Medial Meniscus': 1.0,
        'Medial OA': 0.0,
        'PF OA': 0.0,
        'Synovitis': 0.0,
        'RelevantKeywords': 'medial meniscus bucket handle tear, subchondral bone marrow oedema medial tibial plateau'
    },
    {
        'SequenceID': 37,
        'StudyInstanceUID': '1.2.826.0.1.3680043.8.498.10960770430559425421609885784366918575',
        'DetectedLanguage': 'English',
        'ACL': 0.0,
        "Baker's": 0.5,
        'Contusion': 0.0,
        'Effusion': 0.3,
        'Fracture': 0.0,
        'Lateral Meniscus': 0.0,
        'Lateral OA': 0.0,
        'MCL': 0.0,
        'Medial Meniscus': 1.0,
        'Medial OA': 0.3,
        'PF OA': 0.5,
        'Synovitis': 0.0,
        'RelevantKeywords': 'complete detached radial tear posterior root medial meniscus, mild chondrosis, small joint effusion, small Baker\'s cyst'
    },
    {
        'SequenceID': 38,
        'StudyInstanceUID': '1.2.826.0.1.3680043.8.498.10964287649604862436796341368993301245',
        'DetectedLanguage': 'English',
        'ACL': 0.0,
        "Baker's": 0.0,
        'Contusion': 0.0,
        'Effusion': 0.0,
        'Fracture': 0.0,
        'Lateral Meniscus': 0.0,
        'Lateral OA': 0.0,
        'MCL': 0.5,
        'Medial Meniscus': 1.0,
        'Medial OA': 0.0,
        'PF OA': 0.8,
        'Synovitis': 0.0,
        'RelevantKeywords': 'horizontal tear medial meniscus, chondrosis with subchondral edema and cyst at trochlea, MCL grade I sprain'
    },
    {
        'SequenceID': 39,
        'StudyInstanceUID': '1.2.826.0.1.3680043.8.498.10965573028231658307306828073340279415',
        'DetectedLanguage': 'Spanish',
        'ACL': 0.0,
        "Baker's": 0.0,
        'Contusion': 0.0,
        'Effusion': 0.0,
        'Fracture': 0.0,
        'Lateral Meniscus': 0.0,
        'Lateral OA': 0.5,
        'MCL': 0.0,
        'Medial Meniscus': 0.8,
        'Medial OA': 0.5,
        'PF OA': 0.5,
        'Synovitis': 0.0,
        'RelevantKeywords': 'aumento de señal menisco medial, quiste parameniscal, condropatía focal'
    },
    {
        'SequenceID': 40,
        'StudyInstanceUID': '1.2.826.0.1.3680043.8.498.10975347305662629858093628672419901528',
        'DetectedLanguage': 'Greek',
        'ACL': 0.0,
        "Baker's": 0.0,
        'Contusion': 0.0,
        'Effusion': 0.0,
        'Fracture': 0.0,
        'Lateral Meniscus': 0.0,
        'Lateral OA': 0.0,
        'MCL': 0.0,
        'Medial Meniscus': 0.3,
        'Medial OA': 0.0,
        'PF OA': 0.0,
        'Synovitis': 0.0,
        'RelevantKeywords': 'ήπια εκφύλιση οπίσθιου κέρατος έσω μηνίσκου'
    },
    {
        'SequenceID': 41,
        'StudyInstanceUID': '1.2.826.0.1.3680043.8.498.10979105661314213697886164132444977106',
        'DetectedLanguage': 'English',
        'ACL': 0.0,
        "Baker's": 0.0,
        'Contusion': 0.0,
        'Effusion': 0.3,
        'Fracture': 0.0,
        'Lateral Meniscus': 0.0,
        'Lateral OA': 0.0,
        'MCL': 0.0,
        'Medial Meniscus': 0.0,
        'Medial OA': 0.0,
        'PF OA': 0.0,
        'Synovitis': 0.0,
        'RelevantKeywords': 'minor knee effusion'
    },
    {
        'SequenceID': 42,
        'StudyInstanceUID': '1.2.826.0.1.3680043.8.498.10981062990502278795793220223076241424',
        'DetectedLanguage': 'Croatian',
        'ACL': 0.3,
        "Baker's": 0.0,
        'Contusion': 0.0,
        'Effusion': 0.7,
        'Fracture': 0.0,
        'Lateral Meniscus': 0.5,
        'Lateral OA': 1.0,
        'MCL': 0.0,
        'Medial Meniscus': 1.0,
        'Medial OA': 1.0,
        'PF OA': 1.0,
        'Synovitis': 1.0,
        'RelevantKeywords': 'degen promjene, hondromalacija, koštani edem patele i trohleje, medijalni menisk ruptura i ciste, sinovitis'
    },
    {
        'SequenceID': 43,
        'StudyInstanceUID': '1.2.826.0.1.3680043.8.498.10983416091338854252058678062116243962',
        'DetectedLanguage': 'English',
        'ACL': 1.0,
        "Baker's": 0.3,
        'Contusion': 1.0,
        'Effusion': 0.3,
        'Fracture': 0.0,
        'Lateral Meniscus': 0.0,
        'Lateral OA': 0.0,
        'MCL': 0.5,
        'Medial Meniscus': 0.0,
        'Medial OA': 0.0,
        'PF OA': 0.0,
        'Synovitis': 0.0,
        'RelevantKeywords': 'near-complete to complete tear distal ACL, osseous contusions, trace Baker\'s cyst, MCL grade 1 sprain'
    },
    {
        'SequenceID': 44,
        'StudyInstanceUID': '1.2.826.0.1.3680043.8.498.10992875360571368325079907908221899699',
        'DetectedLanguage': 'English',
        'ACL': 0.8,
        "Baker's": 0.0,
        'Contusion': 0.0,
        'Effusion': 0.7,
        'Fracture': 0.0,
        'Lateral Meniscus': 0.0,
        'Lateral OA': 0.0,
        'MCL': 0.0,
        'Medial Meniscus': 1.0,
        'Medial OA': 0.3,
        'PF OA': 0.0,
        'Synovitis': 0.5,
        'RelevantKeywords': 'complex radial tear medial meniscus, partial tear medial retinaculum, high grade partial ACL tear, moderate effusion with synovial proliferation'
    },
    {
        'SequenceID': 45,
        'StudyInstanceUID': '1.2.826.0.1.3680043.8.498.10994554560803816407362754025763036100',
        'DetectedLanguage': 'German',
        'ACL': 0.0,
        "Baker's": 0.0,
        'Contusion': 0.0,
        'Effusion': 0.0,
        'Fracture': 0.0,
        'Lateral Meniscus': 0.0,
        'Lateral OA': 0.3,
        'MCL': 0.0,
        'Medial Meniscus': 0.0,
        'Medial OA': 0.3,
        'PF OA': 0.5,
        'Synovitis': 0.0,
        'RelevantKeywords': 'retropatellar chondropathische Signalveränderungen, Rissbildung zentral'
    },
    {
        'SequenceID': 46,
        'StudyInstanceUID': '1.2.826.0.1.3680043.8.498.10994848378662155711176898659205016445',
        'DetectedLanguage': 'Spanish',
        'ACL': 1.0,
        "Baker's": 1.0,
        'Contusion': 1.0,
        'Effusion': 0.0,
        'Fracture': 0.0,
        'Lateral Meniscus': 0.0,
        'Lateral OA': 0.0,
        'MCL': 0.0,
        'Medial Meniscus': 0.0,
        'Medial OA': 0.0,
        'PF OA': 0.0,
        'Synovitis': 0.0,
        'RelevantKeywords': 'rotura completa de LCA, contusión ósea femoral, pinzamiento almohadilla grasa Hoffa, quiste de Baker con fuga'
    },
    {
        'SequenceID': 47,
        'StudyInstanceUID': '1.2.826.0.1.3680043.8.498.10994854354532517072359864903910225174',
        'DetectedLanguage': 'Turkish',
        'ACL': 0.0,
        "Baker's": 0.0,
        'Contusion': 0.0,
        'Effusion': 0.7,
        'Fracture': 0.0,
        'Lateral Meniscus': 0.0,
        'Lateral OA': 0.0,
        'MCL': 0.0,
        'Medial Meniscus': 1.0,
        'Medial OA': 1.0,
        'PF OA': 0.5,
        'Synovitis': 0.0,
        'RelevantKeywords': 'trikompartmantal dejeneratif eklem hastalığı, medyal menisküs vertikal yırtık ve ekstrüzyon, eklem içi sıvı artışı'
    },
    {
        'SequenceID': 48,
        'StudyInstanceUID': '1.2.826.0.1.3680043.8.498.10994899269366718796438389394954844979',
        'DetectedLanguage': 'Spanish',
        'ACL': 0.0,
        "Baker's": 0.0,
        'Contusion': 0.0,
        'Effusion': 0.3,
        'Fracture': 0.0,
        'Lateral Meniscus': 0.0,
        'Lateral OA': 0.0,
        'MCL': 0.0,
        'Medial Meniscus': 0.0,
        'Medial OA': 0.0,
        'PF OA': 0.0,
        'Synovitis': 0.0,
        'RelevantKeywords': 'distensión de la bolsa infrarrotuliana profunda'
    },
    {
        'SequenceID': 49,
        'StudyInstanceUID': '1.2.826.0.1.3680043.8.498.10995980880227536691827030731644023480',
        'DetectedLanguage': 'Turkish',
        'ACL': 0.0,
        "Baker's": 0.0,
        'Contusion': 0.0,
        'Effusion': 0.0,
        'Fracture': 0.0,
        'Lateral Meniscus': 0.0,
        'Lateral OA': 0.0,
        'MCL': 0.0,
        'Medial Meniscus': 0.0,
        'Medial OA': 0.0,
        'PF OA': 0.0,
        'Synovitis': 0.0,
        'RelevantKeywords': 'Normal'
    },
    {
        'SequenceID': 50,
        'StudyInstanceUID': '1.2.826.0.1.3680043.8.498.10998144118962608870602180672090998054',
        'DetectedLanguage': 'Turkish',
        'ACL': 0.0,
        "Baker's": 0.0,
        'Contusion': 0.0,
        'Effusion': 0.3,
        'Fracture': 0.0,
        'Lateral Meniscus': 0.0,
        'Lateral OA': 0.0,
        'MCL': 0.0,
        'Medial Meniscus': 1.0,
        'Medial OA': 0.0,
        'PF OA': 0.0,
        'Synovitis': 0.0,
        'RelevantKeywords': 'medial menisküs posterior horn horizontal yırtık, minimal efüzyon'
    }
]

df = pd.DataFrame(data)
df.to_csv('batch_7_output.csv', index=False)
df